# Fluoro / CXR MVP Research Notebook

Safety-first research pipeline for chest screening triage:

- `no_attention_required`
- `requires_attention`
- `N/A / manual_review`

This notebook implements the first branch of two main experiments:

- **EXP-1:** Google CXR Foundation full image + ROI embeddings, shallow calibrated heads.
- **EXP-2:** EVA-X-S frozen encoder + head, with optional LoRA/adapters extension.

Optional **EXP-3 CheXFound** is included as a scaffold/reference path and is off by default.

Product note: this is not an automatic doctor. The notebook tests whether a model can safely auto-clear a controlled subset of studies while sending suspicious or uncertain cases to manual review.

## 0. Run Config

Edit this cell first. It is the single control panel for dataset choice, model branches, sample size, and output location.

Default profile for the first Colab machine:

- IN-CXR only.
- EXP-1 Google CXR Foundation frozen embeddings + calibrated heads.
- EXP-2 EVA-X-S frozen encoder + calibrated heads.
- LoRA is optional and disabled by default.
- VinDr/VinBigData is disabled here and should be run in a separate runtime/profile.

In [ ]:
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB_BOOTSTRAP = True
except Exception:
    IN_COLAB_BOOTSTRAP = False

# Dataset profile.
os.environ["DOWNLOAD_IN_CXR_FROM_KAGGLE"] = "1"
os.environ["RUN_PRIMARY_TRACK"] = "1"
os.environ["MAX_STUDIES"] = "5000"

# Keep the second dataset off on this machine.
os.environ["DOWNLOAD_VINDR_FROM_KAGGLE"] = "0"
os.environ["RUN_VINDR_TRACK"] = "0"
os.environ["RUN_VINDR_EXP2"] = "0"
os.environ["VINDR_DOWNLOAD_MODE"] = "png512"
os.environ["VINDR_DOWNLOAD_MAX_STUDIES"] = "5000"
os.environ["MAX_VINDR_STUDIES"] = "5000"

# Main model branches.
os.environ["RUN_REAL_GOOGLE_CXR"] = "1"
os.environ["RUN_REAL_EVA_X"] = "1"
os.environ["RUN_EXP2_LORA"] = "0"
os.environ["RUN_EXP3_CHEXFOUND"] = "0"

# Download guard defaults. The helper cell only reads these values.
os.environ["DOWNLOAD_VINDR_FULL_FROM_KAGGLE"] = "0"
os.environ["VINDR_KEEP_ZIPS"] = "0"

# Resource-sensitive options.
os.environ["CXR_FOUNDATION_FULL_SIZE"] = "512"
os.environ["EXP2_LORA_EPOCHS"] = "2"
os.environ["EXP2_LORA_BATCH_SIZE"] = "4"
os.environ["EXP2_SELECTED_THRESHOLD_POLICY"] = "zero_fn_cap_08pct"
os.environ["EXP2_SELECTION_OBJECTIVE"] = "quality_first"
os.environ["CACHE_PREPROCESSED_TO_DISK"] = "1"
os.environ["PREPROCESSED_CACHE_DIR"] = "/content/fluoro_mvp_preprocessed_cache" if IN_COLAB_BOOTSTRAP else str(Path.cwd() / "fluoro_mvp_preprocessed_cache")

# Persist outputs to Drive in Colab so a runtime reset does not erase results.
if IN_COLAB_BOOTSTRAP:
    os.environ["MOUNT_DRIVE"] = "1"
    os.environ["PROJECT_DIR"] = "/content/drive/MyDrive/fluoro_mvp_runs/incxr_t4"
else:
    os.environ.setdefault("MOUNT_DRIVE", "0")
    os.environ.setdefault("PROJECT_DIR", str(Path.cwd() / "fluoro_mvp_outputs"))

# If data is already mounted, set these instead of downloading:
# os.environ["IN_CXR_ROOT"] = "/content/path/to/incxr"
# os.environ["IN_CXR_LABELS_CSV"] = "/content/path/to/labels.csv"
# os.environ["VINDR_ROOT"] = "/content/path/to/vindr"

print("Run profile configured:")
for key in [
    "PROJECT_DIR",
    "MAX_STUDIES",
    "DOWNLOAD_IN_CXR_FROM_KAGGLE",
    "RUN_PRIMARY_TRACK",
    "DOWNLOAD_VINDR_FROM_KAGGLE",
    "RUN_VINDR_TRACK",
    "RUN_REAL_GOOGLE_CXR",
    "RUN_REAL_EVA_X",
    "RUN_EXP2_LORA",
    "RUN_EXP3_CHEXFOUND",
    "EXP2_SELECTED_THRESHOLD_POLICY",
    "EXP2_SELECTION_OBJECTIVE",
    "CACHE_PREPROCESSED_TO_DISK",
    "PREPROCESSED_CACHE_DIR",
]:
    print(f"  {key}={os.environ.get(key)}")

## 1. Colab Setup

Run this notebook on a Google Colab GPU runtime. Tesla T4 16GB is enough for the default first-machine path because the heavy encoders are frozen and embeddings are cached.

Default mode is **production IN-CXR mode**. It downloads the IN-CXR Kaggle mirror, uses up to **5000 IN-CXR studies**, preprocesses real images, trains the first-branch heads, calibrates the router, exports reports, and builds interpretation artifacts.

The heavy preprocessed image cache is stored on local Colab disk through `PREPROCESSED_CACHE_DIR=/content/fluoro_mvp_preprocessed_cache`. Google Drive keeps compact reports, embeddings, checkpoints, and final artifacts; it is not used for tens of thousands of temporary image-cache files.

Production run checklist:

1. Add `KAGGLE_API_TOKEN` in Colab Secrets.
2. Accept Hugging Face terms for Google CXR Foundation and add `HF_TOKEN` in Colab Secrets.
3. Run cells from the top. The notebook downloads the bounded IN-CXR subset and sets `IN_CXR_ROOT` automatically.
4. EVA-X-S weights are downloaded from the official Hugging Face checkpoint link used by the EVA-X repo when `RUN_REAL_EVA_X=1`.

For the separate VinDr/VinBigData machine, also accept Kaggle terms for `vinbigdata-chest-xray-abnormalities-detection`.

About tokens:

- `HF_TOKEN` is used for gated Hugging Face models such as Google CXR Foundation after terms are accepted.
- Kaggle can use a modern `KAGGLE_API_TOKEN` access token, or legacy `KAGGLE_USERNAME` + `KAGGLE_KEY` / Kaggle JSON.
- Production defaults enable bounded Kaggle downloads.

First production target: use **5000 IN-CXR studies** for screening behavior. VinDr/VinBigData is intentionally off in this first-machine profile.

In [ ]:
# Install only lightweight dependencies by default.
# Heavy/optional model packages are installed later only when the corresponding real-model flag is enabled.
import importlib.util
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

from packaging.version import Version

os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")

def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

def pip_install(package: str) -> None:
    print(f"Installing {package} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

def pip_install_upgrade(package: str) -> None:
    print(f"Upgrading {package} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", package], check=True)

def ensure_distribution_min_version(distribution: str, package_spec: str, min_version: str) -> None:
    try:
        current = version(distribution)
    except PackageNotFoundError:
        pip_install(package_spec)
        return
    if Version(current) < Version(min_version):
        pip_install_upgrade(package_spec)

BASE_DEPS = [
    ("pydicom", "pydicom>=2.4.4"),
    ("joblib", "joblib>=1.3.0"),
    ("scipy", "scipy>=1.10.0"),
    ("sklearn", "scikit-learn>=1.2.0"),
    ("PIL", "Pillow>=9.5.0"),
    ("tqdm", "tqdm>=4.66.0"),
    ("matplotlib", "matplotlib>=3.7.0"),
    ("huggingface_hub", "huggingface_hub>=0.23.0"),
    ("cloudpickle", "cloudpickle>=2.2.0"),
]

if os.environ.get("FLUORO_SKIP_INSTALLS", "0") != "1":
    for import_name, package in BASE_DEPS:
        if not has_module(import_name):
            pip_install(package)
    ensure_distribution_min_version("ml-dtypes", "ml-dtypes>=0.5.0", "0.5.0")

print("Setup cell finished.")

## 2. Tokens and Dataset Download Helpers

This cell reads Colab Secrets, prepares Kaggle/Hugging Face authentication, and downloads only the datasets enabled in **Run Config**.

Recommended Colab pattern:

```python
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN") or ""
```

Legacy Kaggle username/key JSON is also supported:

```python
os.environ["KAGGLE_API_TOKEN"] = '{"username":"...","key":"..."}'
```

Current first-machine defaults used by the notebook:

```python
os.environ["DOWNLOAD_IN_CXR_FROM_KAGGLE"] = "1"
os.environ["DOWNLOAD_VINDR_FROM_KAGGLE"] = "0"
os.environ["MAX_STUDIES"] = "5000"
os.environ["VINDR_DOWNLOAD_MAX_STUDIES"] = "5000"
os.environ["MAX_VINDR_STUDIES"] = "5000"
os.environ["VINDR_DOWNLOAD_MODE"] = "png512"
os.environ["CXR_FOUNDATION_FULL_SIZE"] = "512"
os.environ["RUN_REAL_GOOGLE_CXR"] = "1"
os.environ["RUN_REAL_EVA_X"] = "1"
os.environ["RUN_PRIMARY_TRACK"] = "1"
os.environ["RUN_VINDR_TRACK"] = "0"
os.environ["RUN_VINDR_EXP2"] = "0"
os.environ["RUN_EXP2_LORA"] = "0"
os.environ["EXP2_LORA_EPOCHS"] = "2"
os.environ["EXP2_LORA_BATCH_SIZE"] = "4"
os.environ["EXP2_SELECTED_THRESHOLD_POLICY"] = "zero_fn_cap_08pct"
os.environ["EXP2_SELECTION_OBJECTIVE"] = "quality_first"
os.environ["MOUNT_DRIVE"] = "1"
os.environ["PROJECT_DIR"] = "/content/drive/MyDrive/fluoro_mvp_runs/incxr_t4"
os.environ["PREPROCESSED_CACHE_DIR"] = "/content/fluoro_mvp_preprocessed_cache"
os.environ["RUN_EXP3_CHEXFOUND"] = "0"
```

The IN-CXR helper downloads the Kaggle PNG mirror and sets `IN_CXR_ROOT`. The VinDr helper stays idle in this profile; if enabled later, it downloads `train.csv`, samples image IDs, then downloads only selected PNG512 files plus original DICOM metadata needed to scale bboxes.

Resource profile for a single Colab runtime:

```python
# IN-CXR only: lighter first run
os.environ["DOWNLOAD_VINDR_FROM_KAGGLE"] = "0"
os.environ["RUN_VINDR_TRACK"] = "0"
```

The default first-machine profile is IN-CXR only. Use a separate notebook copy/profile for the VinDr machine so the two runs do not compete for Colab disk/RAM.

Note: the Kaggle IN-CXR mirror is preprocessed 224x224 PNG, not the official original DICOM release.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import pandas as pd

def get_colab_secret(name: str):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value if value else None
    except Exception:
        return None

def setup_hf_token():
    token = (
        os.environ.get("HF_TOKEN")
        or os.environ.get("HF_TOKET")
        or get_colab_secret("HF_TOKEN")
        or get_colab_secret("HF_TOKET")
    )
    if token:
        os.environ["HF_TOKEN"] = token
        try:
            from huggingface_hub import login
            login(token=token)
            print("HF token configured.")
        except Exception as exc:
            print("HF token present, but login skipped/failed:", exc)
    else:
        print("HF_TOKEN not set. Real gated HF models will not load until you set it.")

def setup_kaggle_token():
    username = os.environ.get("KAGGLE_USERNAME") or get_colab_secret("KAGGLE_USERNAME")
    key = os.environ.get("KAGGLE_KEY") or get_colab_secret("KAGGLE_KEY")
    token_value = (
        os.environ.get("KAGGLE_API_TOKEN")
        or os.environ.get("KAGGLE_JSON")
        or get_colab_secret("KAGGLE_API_TOKEN")
        or get_colab_secret("KAGGLE_JSON")
    )

    if token_value:
        token_value = str(token_value).strip()

    if token_value and token_value.startswith("{") and not (username and key):
        creds = json.loads(token_value)
        username = creds.get("username")
        key = creds.get("key")
        # The Kaggle CLI treats KAGGLE_API_TOKEN as an access token first,
        # so remove JSON-shaped content after converting it to legacy env vars.
        if os.environ.get("KAGGLE_API_TOKEN", "").strip().startswith("{"):
            os.environ.pop("KAGGLE_API_TOKEN", None)

    if username and key:
        os.environ["KAGGLE_USERNAME"] = username
        os.environ["KAGGLE_KEY"] = key
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(exist_ok=True)
        kaggle_json = kaggle_dir / "kaggle.json"
        kaggle_json.write_text(json.dumps({
            "username": os.environ["KAGGLE_USERNAME"],
            "key": os.environ["KAGGLE_KEY"],
        }), encoding="utf-8")
        kaggle_json.chmod(0o600)
        print("Kaggle legacy credentials configured.")
        return True

    if token_value:
        os.environ["KAGGLE_API_TOKEN"] = token_value
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(exist_ok=True)
        access_token_file = kaggle_dir / "access_token"
        access_token_file.write_text(token_value, encoding="utf-8")
        access_token_file.chmod(0o600)
        print("Kaggle access token configured from KAGGLE_API_TOKEN.")
        return True

    print("Kaggle credentials not set. Kaggle download helper will be skipped.")
    return False

def kaggle_cli_executable() -> str:
    exe = shutil.which("kaggle")
    if exe is None:
        raise RuntimeError("Kaggle CLI executable was not found after installation.")
    return exe

def maybe_download_kaggle_competition(
    competition: str,
    target_dir: str | Path,
    unzip: bool = True,
    env_root_var: str | None = None,
):
    if not setup_kaggle_token():
        raise RuntimeError("Set Kaggle credentials before downloading from Kaggle.")
    if not has_module("kaggle"):
        pip_install("kaggle")
    elif os.environ.get("KAGGLE_API_TOKEN") and not os.environ.get("KAGGLE_USERNAME"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)
    kaggle_cli = kaggle_cli_executable()

    target = Path(target_dir)
    target.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [kaggle_cli, "competitions", "download", "-c", competition, "-p", str(target)],
        check=True,
    )

    if unzip:
        zip_paths = sorted(target.glob("*.zip"))
        if not zip_paths:
            print("No zip archives found after Kaggle download.")
        for zip_path in zip_paths:
            print("Unzipping:", zip_path.name)
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(target)

    sample_files = [p for p in target.rglob("*") if p.is_file()][:20]
    print("Kaggle data ready at:", target)
    for p in sample_files:
        print(" -", p.relative_to(target))

    if env_root_var:
        os.environ[env_root_var] = str(target)
        print(f"{env_root_var}={target}")
    return str(target)

def download_kaggle_competition_file(competition: str, filename: str, target: Path, env: dict | None = None) -> None:
    kaggle_cli = kaggle_cli_executable()
    subprocess.run(
        [kaggle_cli, "competitions", "download", "-c", competition, "-f", filename, "-p", str(target)],
        check=True,
        env=env,
    )
    for zip_path in sorted(target.glob("*.zip")):
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(target)
        if os.environ.get("VINDR_KEEP_ZIPS", "0") != "1":
            zip_path.unlink()

def download_kaggle_dataset_file(dataset: str, filename: str | None, target: Path, env: dict | None = None) -> None:
    kaggle_cli = kaggle_cli_executable()
    cmd = [kaggle_cli, "datasets", "download", "-d", dataset, "-p", str(target)]
    if filename:
        cmd.extend(["-f", filename])
    subprocess.run(cmd, check=True, env=env)
    for zip_path in sorted(target.glob("*.zip")):
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(target)
        if os.environ.get("VINDR_KEEP_ZIPS", "0") != "1":
            zip_path.unlink()

def sample_vindr_ids_from_train_csv(train_csv: Path, max_studies: int, normal_fraction: float = 0.50, seed: int = 42) -> pd.DataFrame:
    labels = pd.read_csv(train_csv)
    labels.columns = [c.strip() for c in labels.columns]
    if "image_id" not in labels.columns or "class_name" not in labels.columns:
        raise ValueError("VinDr/VinBigData train.csv must contain image_id and class_name columns.")
    tmp = labels[["image_id", "class_name"]].copy()
    tmp["image_id"] = tmp["image_id"].astype(str)
    tmp["is_abnormal_row"] = tmp["class_name"].fillna("").astype(str).str.lower().ne("no finding")
    image_level = tmp.groupby("image_id", as_index=False)["is_abnormal_row"].max()
    image_level["y_attention"] = image_level["is_abnormal_row"].astype(int)
    normal = image_level[image_level["y_attention"] == 0]
    abnormal = image_level[image_level["y_attention"] == 1]
    n_normal = min(len(normal), int(round(max_studies * normal_fraction)))
    n_abnormal = min(len(abnormal), max_studies - n_normal)
    if n_abnormal < max_studies - n_normal and len(normal) > n_normal:
        n_normal = min(len(normal), max_studies - n_abnormal)
    normal_s = normal.sample(n=n_normal, random_state=seed) if n_normal else normal.head(0)
    abnormal_s = abnormal.sample(n=n_abnormal, random_state=seed) if n_abnormal else abnormal.head(0)
    selected = pd.concat([normal_s, abnormal_s], ignore_index=True).sample(frac=1.0, random_state=seed)
    return selected[["image_id", "y_attention"]].reset_index(drop=True)

def maybe_download_vindr_subset_from_kaggle(target_dir="/content/vindr_cxr", max_studies: int | None = None):
    if not setup_kaggle_token():
        raise RuntimeError("Set Kaggle credentials before downloading VinDr from Kaggle.")
    if not has_module("kaggle"):
        pip_install("kaggle")
    elif os.environ.get("KAGGLE_API_TOKEN") and not os.environ.get("KAGGLE_USERNAME"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)

    target = Path(target_dir)
    target.mkdir(parents=True, exist_ok=True)
    competition = os.environ.get("VINDR_KAGGLE_COMPETITION", "vinbigdata-chest-xray-abnormalities-detection")
    max_studies = int(max_studies or os.environ.get("VINDR_DOWNLOAD_MAX_STUDIES", "5000"))
    normal_fraction = float(os.environ.get("VINDR_DOWNLOAD_NORMAL_FRACTION", "0.50"))
    seed = int(os.environ.get("VINDR_DOWNLOAD_SEED", "42"))

    train_csv = target / "train.csv"
    if not train_csv.exists():
        print("Downloading VinDr/VinBigData train.csv ...")
        download_kaggle_competition_file(competition, "train.csv", target, env=os.environ.copy())

    selected = sample_vindr_ids_from_train_csv(train_csv, max_studies=max_studies, normal_fraction=normal_fraction, seed=seed)
    manifest_path = target / "vindr_subset_manifest.csv"
    selected.to_csv(manifest_path, index=False)
    print(f"Selected {len(selected)} studies:", selected["y_attention"].value_counts().to_dict())
    print("Subset manifest:", manifest_path)

    for idx, row in selected.iterrows():
        image_id = str(row["image_id"])
        dicom_path = target / f"{image_id}.dicom"
        nested_dicom_path = target / "train" / f"{image_id}.dicom"
        if dicom_path.exists() or nested_dicom_path.exists():
            continue
        if idx % 25 == 0:
            print(f"Downloading DICOM {idx + 1}/{len(selected)} ...")
        download_kaggle_competition_file(competition, f"train/{image_id}.dicom", target, env=os.environ.copy())

    os.environ["VINDR_ROOT"] = str(target)
    print("VinDr subset ready at:", target)
    print("VINDR_ROOT=", target)
    return str(target)

def maybe_download_vindr_png_subset_from_kaggle(target_dir="/content/vindr_cxr", max_studies: int | None = None):
    if not setup_kaggle_token():
        raise RuntimeError("Set Kaggle credentials before downloading VinDr from Kaggle.")
    if not has_module("kaggle"):
        pip_install("kaggle")
    elif os.environ.get("KAGGLE_API_TOKEN") and not os.environ.get("KAGGLE_USERNAME"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)

    target = Path(target_dir)
    target.mkdir(parents=True, exist_ok=True)
    competition = os.environ.get("VINDR_KAGGLE_COMPETITION", "vinbigdata-chest-xray-abnormalities-detection")
    image_dataset = os.environ.get("VINDR_KAGGLE_PNG_DATASET", "xhlulu/vinbigdata")
    metadata_dataset = os.environ.get("VINDR_KAGGLE_METADATA_DATASET", "sunhwan/vinbigdata-chest-xray-dicom-metadata")
    image_ext = os.environ.get("VINDR_KAGGLE_IMAGE_EXT", "png")
    max_studies = int(max_studies or os.environ.get("VINDR_DOWNLOAD_MAX_STUDIES", "5000"))
    normal_fraction = float(os.environ.get("VINDR_DOWNLOAD_NORMAL_FRACTION", "0.50"))
    seed = int(os.environ.get("VINDR_DOWNLOAD_SEED", "42"))

    train_csv = target / "train.csv"
    if not train_csv.exists():
        print("Downloading VinDr/VinBigData train.csv ...")
        download_kaggle_competition_file(competition, "train.csv", target, env=os.environ.copy())

    if not any((target / name).exists() for name in ["images.csv", "train_meta.csv", "metadata.csv", "dicom_metadata.csv"]):
        print("Downloading original DICOM metadata for bbox scaling ...")
        download_kaggle_dataset_file(metadata_dataset, None, target, env=os.environ.copy())

    selected = sample_vindr_ids_from_train_csv(train_csv, max_studies=max_studies, normal_fraction=normal_fraction, seed=seed)
    manifest_path = target / "vindr_subset_manifest.csv"
    selected.to_csv(manifest_path, index=False)
    print(f"Selected {len(selected)} studies:", selected["y_attention"].value_counts().to_dict())
    print("Subset manifest:", manifest_path)

    for idx, row in selected.iterrows():
        image_id = str(row["image_id"])
        image_path = target / f"{image_id}.{image_ext}"
        nested_image_path = target / "train" / f"{image_id}.{image_ext}"
        if image_path.exists() or nested_image_path.exists():
            continue
        if idx % 100 == 0:
            print(f"Downloading PNG {idx + 1}/{len(selected)} ...")
        download_kaggle_dataset_file(image_dataset, f"train/{image_id}.{image_ext}", target, env=os.environ.copy())

    os.environ["VINDR_ROOT"] = str(target)
    print("VinDr PNG subset ready at:", target)
    print("VINDR_ROOT=", target)
    return str(target)

def maybe_download_vindr_from_kaggle(target_dir="/content/vindr_cxr"):
    if os.environ.get("DOWNLOAD_VINDR_FROM_KAGGLE", "0") != "1":
        print("DOWNLOAD_VINDR_FROM_KAGGLE=0, skipping Kaggle download.")
        return None
    if os.environ.get("DOWNLOAD_VINDR_FULL_FROM_KAGGLE", "0") != "1":
        mode = os.environ.get("VINDR_DOWNLOAD_MODE", "png512").lower()
        if mode in {"png", "png512", "png_512", "512"}:
            return maybe_download_vindr_png_subset_from_kaggle(target_dir=target_dir)
        if mode in {"png256", "png_256", "256"}:
            os.environ.setdefault("VINDR_KAGGLE_PNG_DATASET", "xhlulu/vinbigdata-chest-xray-resized-png-256x256")
            return maybe_download_vindr_png_subset_from_kaggle(target_dir=target_dir)
        if mode == "dicom":
            return maybe_download_vindr_subset_from_kaggle(target_dir=target_dir)
        raise ValueError(f"Unknown VINDR_DOWNLOAD_MODE={mode!r}; use png512, png256, or dicom.")
    competition = os.environ.get("VINDR_KAGGLE_COMPETITION", "vinbigdata-chest-xray-abnormalities-detection")
    return maybe_download_kaggle_competition(
        competition=competition,
        target_dir=target_dir,
        unzip=True,
        env_root_var="VINDR_ROOT",
    )

def maybe_download_incxr_from_kaggle(target_dir="/content/incxr_png"):
    if os.environ.get("DOWNLOAD_IN_CXR_FROM_KAGGLE", "0") != "1":
        print("DOWNLOAD_IN_CXR_FROM_KAGGLE=0, skipping IN-CXR Kaggle download.")
        return None
    if not setup_kaggle_token():
        raise RuntimeError("Set Kaggle credentials before downloading IN-CXR from Kaggle.")
    if not has_module("kaggle"):
        pip_install("kaggle")
    elif os.environ.get("KAGGLE_API_TOKEN") and not os.environ.get("KAGGLE_USERNAME"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)

    target = Path(target_dir)
    target.mkdir(parents=True, exist_ok=True)
    dataset = os.environ.get("IN_CXR_KAGGLE_DATASET", "arjav007/in-cxr-dataset-png")
    marker = target / ".incxr_kaggle_download_complete"
    if not marker.exists():
        print("Downloading IN-CXR Kaggle PNG mirror. Expected compressed size is about 380 MB.")
        download_kaggle_dataset_file(dataset, None, target, env=os.environ.copy())
        marker.write_text(dataset, encoding="utf-8")
    else:
        print("IN-CXR Kaggle mirror already downloaded:", target)

    sample_files = [p for p in target.rglob("*.png")][:10]
    print("IN-CXR PNG files found:", len(list(target.rglob("*.png"))))
    for p in sample_files:
        print(" -", p.relative_to(target))
    os.environ["IN_CXR_ROOT"] = str(target)
    print("IN_CXR_ROOT=", target)
    return str(target)

setup_hf_token()
setup_kaggle_token()
if os.environ.get("RUN_PRIMARY_TRACK", "1") == "1":
    maybe_download_incxr_from_kaggle(os.environ.get("IN_CXR_DOWNLOAD_DIR", "/content/incxr_png"))
if os.environ.get("RUN_VINDR_TRACK", "0") == "1":
    maybe_download_vindr_from_kaggle(os.environ.get("VINDR_DOWNLOAD_DIR", "/content/vindr_cxr"))

## 3. Core Pipeline Code

The next cell contains the self-contained implementation used by the notebook: dataset discovery, DICOM/PNG preprocessing, ROI preparation, feature extraction helpers, model heads, calibration, routing, metrics, LoRA utilities, and quantization helpers.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import subprocess
import sys
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable

import joblib
import numpy as np
import pandas as pd
from PIL import Image

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception:  # pragma: no cover - notebook install cell handles this in Colab.
    torch = None
    nn = None
    F = None

try:
    import pydicom
except Exception:  # pragma: no cover - pydicom is optional until DICOM cells run.
    pydicom = None

from scipy import ndimage
from sklearn.base import BaseEstimator
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


@dataclass
class NotebookConfig:
    project_dir: str = "/content/fluoro_mvp"
    data_root: str | None = None
    labels_csv: str | None = None
    vindr_root: str | None = None
    max_studies: int | None = 5000
    max_vindr_studies: int | None = 5000
    random_state: int = 42
    target_npv: float = 0.99
    max_fn_per_1000: float = 5.0
    cxr_foundation_full_size: int = 1024
    eva_image_size: int = 224
    batch_size: int = 8
    run_real_google_cxr: bool = True
    run_real_eva_x: bool = True
    run_exp2_lora: bool = False
    run_primary_track: bool = True
    run_vindr_track: bool = True
    run_vindr_exp2: bool = True
    run_exp3_chexfound: bool = False
    pca_components: int | None = 256
    cache_preprocessed_to_disk: bool = True
    preprocessed_cache_dir: str | None = None
    preprocess_progress_every: int = 250
    device: str = "cuda"
    artifacts_dir: str = field(init=False)
    reports_dir: str = field(init=False)
    checkpoints_dir: str = field(init=False)

    def __post_init__(self) -> None:
        if torch is None or not torch.cuda.is_available():
            self.device = "cpu"
        self.artifacts_dir = str(Path(self.project_dir) / "artifacts")
        self.reports_dir = str(Path(self.project_dir) / "reports")
        self.checkpoints_dir = str(Path(self.project_dir) / "checkpoints")
        if self.preprocessed_cache_dir is None:
            if Path("/content").exists():
                self.preprocessed_cache_dir = "/content/fluoro_mvp_preprocessed_cache"
            else:
                self.preprocessed_cache_dir = str(Path(self.project_dir) / "preprocessed_cache")


@dataclass
class PreprocessResult:
    study_id: str
    source_path: str
    y_attention: int
    source_dataset: str
    image_full: Image.Image | None
    image_roi: Image.Image | None
    image_eva: Image.Image | None
    image_raw_preview: np.ndarray | None
    lung_mask: np.ndarray | None
    quality_score: float
    qa_flags: list[str]
    metadata: dict[str, Any]
    preprocess_log: list[str]
    preprocessing_version: str = "preproc_v1"
    source_type: str = "unknown"
    roi_status: str = "not_available"
    critical_qa: bool = False
    image_full_path: str | None = None
    image_roi_path: str | None = None
    image_eva_path: str | None = None
    image_raw_preview_path: str | None = None
    lung_mask_path: str | None = None

    def to_record(self) -> dict[str, Any]:
        record = {
            "study_id": self.study_id,
            "source_path": self.source_path,
            "y_attention": int(self.y_attention),
            "source_dataset": self.source_dataset,
            "quality_score": float(self.quality_score),
            "qa_flags": "|".join(self.qa_flags),
            "preprocess_log": "|".join(self.preprocess_log),
            "preprocessing_version": self.preprocessing_version,
            "source_type": self.source_type,
            "roi_status": self.roi_status,
            "critical_qa": bool(self.critical_qa),
            "image_full_path": self.image_full_path,
            "image_roi_path": self.image_roi_path,
            "image_eva_path": self.image_eva_path,
            "image_raw_preview_path": self.image_raw_preview_path,
            "lung_mask_path": self.lung_mask_path,
        }
        for key in [
            "modality",
            "view_position",
            "photometric_interpretation",
            "bits_stored",
            "manufacturer",
            "rows",
            "columns",
            "patient_id_hash",
            "study_uid_hash",
        ]:
            record[key] = self.metadata.get(key)
        return record


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


def ensure_dirs(cfg: NotebookConfig) -> None:
    for path in [
        cfg.project_dir,
        cfg.artifacts_dir,
        cfg.reports_dir,
        cfg.checkpoints_dir,
        Path(cfg.artifacts_dir) / "previews",
        Path(cfg.artifacts_dir) / "embeddings",
        Path(cfg.artifacts_dir) / "models",
        Path(cfg.artifacts_dir) / "calibration",
        Path(cfg.artifacts_dir) / "router",
        Path(cfg.artifacts_dir) / "quantization",
        cfg.preprocessed_cache_dir,
    ]:
        if path is None:
            continue
        Path(path).mkdir(parents=True, exist_ok=True)


def stable_hash(text: str, n: int = 16) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:n]


def save_table(df: pd.DataFrame, path_no_ext: str | Path) -> str:
    path_no_ext = Path(path_no_ext)
    try:
        path = path_no_ext.with_suffix(".parquet")
        df.to_parquet(path, index=False)
    except Exception:
        path = path_no_ext.with_suffix(".csv")
        df.to_csv(path, index=False)
    return str(path)


def save_pickle(obj: Any, path: str | Path) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        import cloudpickle

        with open(path, "wb") as f:
            cloudpickle.dump(obj, f)
    except Exception:
        joblib.dump(obj, path)
    return str(path)


def _find_first_existing(root: Path, names: list[str]) -> Path | None:
    lower_to_path = {p.name.lower(): p for p in root.rglob("*") if p.is_file()}
    for name in names:
        hit = lower_to_path.get(name.lower())
        if hit is not None:
            return hit
    return None


def _image_id_from_path(path: str | Path) -> str:
    return Path(path).stem


def discover_vindr_dataset(
    vindr_root: str | Path | None,
    max_studies: int | None = 5000,
    cfg: NotebookConfig | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load VinDr-CXR-like DICOM dataset and bbox annotations.

    Supports PhysioNet folder names and Kaggle competition-like CSV layout.
    Returns image-level dataframe plus bbox dataframe.
    """
    if vindr_root is None:
        raise ValueError("Set VINDR_ROOT before running the VinDr track.")
    root = Path(vindr_root)
    if not root.exists():
        raise FileNotFoundError(f"VinDr root not found: {root}")
    image_paths = []
    for pattern in ["*.dicom", "*.dcm", "*.png", "*.jpg", "*.jpeg"]:
        image_paths.extend(root.rglob(pattern))
    path_by_id = {_image_id_from_path(p): str(p) for p in image_paths}
    if not path_by_id:
        raise ValueError(f"No images found under {root}")

    annotation_files = []
    for filename in ["annotations_train.csv", "annotations_test.csv", "train.csv", "test.csv"]:
        p = _find_first_existing(root, [filename])
        if p is not None:
            annotation_files.append(p)
    image_label_files = []
    for filename in ["image_labels_train.csv", "image_labels_test.csv"]:
        p = _find_first_existing(root, [filename])
        if p is not None:
            image_label_files.append(p)
    metadata_files = []
    for filename in ["images.csv", "train_meta.csv", "test_meta.csv", "metadata.csv", "dicom_metadata.csv"]:
        p = _find_first_existing(root, [filename])
        if p is not None:
            metadata_files.append(p)

    original_dim_map: dict[str, tuple[float, float]] = {}
    for p in metadata_files:
        tmp = pd.read_csv(p)
        cols = {c.lower(): c for c in tmp.columns}
        image_id_col = cols.get("image_id")
        if image_id_col is None and "fname" in cols:
            image_id_col = cols["fname"]
        rows_col = cols.get("rows") or cols.get("height") or cols.get("original_height")
        cols_col = cols.get("columns") or cols.get("width") or cols.get("original_width")
        if image_id_col is None or rows_col is None or cols_col is None:
            continue
        for _, meta_row in tmp.iterrows():
            image_id = _image_id_from_path(meta_row[image_id_col])
            rows_value = pd.to_numeric(pd.Series([meta_row[rows_col]]), errors="coerce").iloc[0]
            cols_value = pd.to_numeric(pd.Series([meta_row[cols_col]]), errors="coerce").iloc[0]
            if pd.notna(rows_value) and pd.notna(cols_value) and float(rows_value) > 0 and float(cols_value) > 0:
                original_dim_map[str(image_id)] = (float(rows_value), float(cols_value))

    bbox_frames = []
    for p in annotation_files:
        tmp = pd.read_csv(p)
        tmp.columns = [c.strip() for c in tmp.columns]
        if "image_id" not in tmp.columns:
            continue
        if "class_name" not in tmp.columns and "class_id" in tmp.columns:
            tmp["class_name"] = tmp["class_id"].astype(str)
        for col in ["x_min", "y_min", "x_max", "y_max"]:
            if col not in tmp.columns:
                tmp[col] = np.nan
        tmp["study_id"] = tmp["image_id"].astype(str)
        tmp["source_split"] = "test" if "test" in p.name.lower() else "train"
        bbox_frames.append(tmp[["image_id", "study_id", "class_name", "x_min", "y_min", "x_max", "y_max", "source_split"] + ([c for c in ["rad_id", "rad_ID"] if c in tmp.columns])])
    bboxes = pd.concat(bbox_frames, ignore_index=True) if bbox_frames else pd.DataFrame(
        columns=["image_id", "study_id", "class_name", "x_min", "y_min", "x_max", "y_max", "source_split"]
    )
    if original_dim_map and not bboxes.empty:
        bboxes["bbox_original_rows"] = bboxes["image_id"].astype(str).map(
            lambda x: original_dim_map.get(x, (np.nan, np.nan))[0]
        )
        bboxes["bbox_original_columns"] = bboxes["image_id"].astype(str).map(
            lambda x: original_dim_map.get(x, (np.nan, np.nan))[1]
        )

    label_map: dict[str, int] = {}
    if image_label_files:
        for p in image_label_files:
            labels = pd.read_csv(p)
            labels.columns = [c.strip() for c in labels.columns]
            if "image_id" not in labels.columns:
                continue
            label_cols = [c for c in labels.columns if c not in {"image_id", "rad_id", "rad_ID"}]
            for image_id, part in labels.groupby("image_id"):
                if "No finding" in label_cols:
                    abnormal = bool((part.drop(columns=[c for c in ["image_id", "rad_id", "rad_ID"] if c in part.columns]).sum(axis=1) > part.get("No finding", 0)).any())
                else:
                    numeric = part[label_cols].select_dtypes(include=[np.number])
                    abnormal = bool((numeric.sum(axis=1) > 0).any()) if not numeric.empty else False
                label_map[str(image_id)] = int(abnormal)
    if bboxes is not None and not bboxes.empty:
        for image_id, part in bboxes.groupby("image_id"):
            class_names = part["class_name"].fillna("").astype(str).str.lower()
            has_box = part[["x_min", "y_min", "x_max", "y_max"]].notna().all(axis=1)
            abnormal = bool(((class_names != "no finding") & has_box).any())
            label_map[str(image_id)] = max(label_map.get(str(image_id), 0), int(abnormal))

    rows = []
    for image_id, path in path_by_id.items():
        if image_id not in label_map:
            continue
        original_rows, original_columns = original_dim_map.get(str(image_id), (np.nan, np.nan))
        source_split = "test" if "test" in Path(path).parts else "train"
        rows.append(
            {
                "study_id": image_id,
                "path": path,
                "label_text": "abnormal" if label_map[image_id] else "normal",
                "y_attention": int(label_map[image_id]),
                "source_dataset": "vindr_cxr",
                "patient_id_hash": stable_hash(image_id),
                "official_split": source_split,
                "bbox_original_rows": original_rows,
                "bbox_original_columns": original_columns,
            }
        )
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No VinDr images could be matched to labels/annotations.")
    if max_studies is not None and len(df) > max_studies:
        df = df.groupby("y_attention", group_keys=False).sample(
            frac=min(1.0, max_studies / len(df)), random_state=42
        )
        if len(df) > max_studies:
            df = df.sample(max_studies, random_state=42)
    selected_ids = set(df["study_id"].astype(str))
    bboxes = bboxes[bboxes["study_id"].astype(str).isin(selected_ids)].reset_index(drop=True) if not bboxes.empty else bboxes
    return df.reset_index(drop=True), bboxes


def _normalize_label_token(value: Any) -> str:
    return str(value).strip().lower().replace("-", "_").replace(" ", "_")


def infer_label_from_path(path: str | Path) -> int | None:
    parts = []
    for part in Path(path).parts:
        parts.append(_normalize_label_token(part))
    stem = _normalize_label_token(Path(path).stem)
    parts.append(stem)
    normal_words = {"normal", "no_finding", "no_findings", "negative", "no_attention_required", "healthy", "clear"}
    abnormal_words = {
        "abnormal",
        "positive",
        "requires_attention",
        "attention_required",
        "tb",
        "suspicious",
        "pathology",
        "pathological",
        "disease",
        "diseased",
    }
    if any(part in abnormal_words for part in parts):
        return 1
    if any(part in normal_words for part in parts):
        return 0
    if any(f"_{word}" in stem or f"{word}_" in stem or stem.endswith(word) for word in abnormal_words):
        return 1
    if any(f"_{word}" in stem or f"{word}_" in stem or stem.endswith(word) for word in normal_words):
        return 0
    return None


def label_value_to_attention(value: Any) -> int | None:
    if pd.isna(value):
        return None
    if isinstance(value, (bool, np.bool_)):
        return int(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)) and np.isfinite(value) and float(value).is_integer():
        return int(value)
    text = str(value).strip()
    try:
        numeric = float(text)
        if np.isfinite(numeric) and numeric.is_integer():
            return int(numeric)
    except ValueError:
        pass
    return infer_label_from_path(text)


def discover_dataset(
    data_root: str | Path | None,
    labels_csv: str | Path | None = None,
    max_studies: int | None = None,
    cfg: NotebookConfig | None = None,
) -> pd.DataFrame:
    if labels_csv:
        labels = pd.read_csv(labels_csv)
        path_col = "path" if "path" in labels.columns else "image_path"
        label_col = "y_attention" if "y_attention" in labels.columns else "label"
        rows = []
        for _, row in labels.iterrows():
            p = Path(row[path_col])
            if not p.is_absolute() and data_root:
                p = Path(data_root) / p
            y = label_value_to_attention(row[label_col])
            if y not in {0, 1}:
                raise ValueError(f"Could not infer binary y_attention from label value: {row[label_col]!r}")
            rows.append(
                {
                    "study_id": row.get("study_id", stable_hash(p)),
                    "path": str(p),
                    "label_text": row[label_col],
                    "y_attention": int(y),
                    "source_dataset": row.get("source_dataset", "incxr_or_local"),
                    "patient_id_hash": row.get("patient_id_hash", stable_hash(p.stem)),
                }
            )
        df = pd.DataFrame(rows)
    else:
        if data_root is None:
            raise ValueError("Set data_root or labels_csv before running the IN-CXR track.")
        data_root = Path(data_root)
        patterns = ["*.dcm", "*.dicom", "*.png", "*.jpg", "*.jpeg"]
        paths: list[Path] = []
        for pattern in patterns:
            paths.extend(data_root.rglob(pattern))
        rows = []
        for path in sorted(paths):
            y = infer_label_from_path(path)
            if y is None:
                continue
            rows.append(
                {
                    "study_id": stable_hash(path),
                    "path": str(path),
                    "label_text": "abnormal" if y else "normal",
                    "y_attention": int(y),
                    "source_dataset": "incxr_or_local",
                    "patient_id_hash": stable_hash(path.stem),
                }
            )
        df = pd.DataFrame(rows)
    if max_studies is not None and len(df) > max_studies:
        df = df.groupby("y_attention", group_keys=False).sample(
            frac=min(1.0, max_studies / len(df)), random_state=42
        )
        if len(df) > max_studies:
            df = df.sample(max_studies, random_state=42)
    if df.empty:
        raise ValueError("No labeled images discovered. Expected normal/abnormal folders or labels_csv.")
    return df.reset_index(drop=True)


def make_splits(df: pd.DataFrame, seed: int = 42, group_col: str = "patient_id_hash") -> pd.DataFrame:
    df = df.copy()
    y = df["y_attention"].astype(int)
    if len(df) < 12 or y.nunique() < 2 or y.value_counts().min() < 4:
        df["split"] = "train"
        return df
    if group_col in df.columns and df[group_col].notna().all():
        group_df = (
            df.groupby(group_col, as_index=False)
            .agg(y_attention=("y_attention", "max"), n_images=("y_attention", "size"))
            .reset_index(drop=True)
        )
        gy = group_df["y_attention"].astype(int)
        if len(group_df) >= 12 and gy.nunique() == 2 and gy.value_counts().min() >= 4:
            g_train, g_temp = train_test_split(
                group_df.index,
                test_size=0.40,
                random_state=seed,
                stratify=gy,
            )
            temp_y = gy.loc[g_temp]
            if temp_y.value_counts().min() >= 3:
                g_calib, g_hold = train_test_split(
                    g_temp,
                    test_size=0.50,
                    random_state=seed,
                    stratify=temp_y,
                )
                hold_y = gy.loc[g_hold]
                if hold_y.value_counts().min() >= 2:
                    g_val, g_test = train_test_split(
                        g_hold,
                        test_size=0.50,
                        random_state=seed,
                        stratify=hold_y,
                    )
                else:
                    g_val, g_test = g_hold, []
                df["split"] = "train"
                group_to_split = {}
                for idx in g_train:
                    group_to_split[group_df.loc[idx, group_col]] = "train"
                for idx in g_calib:
                    group_to_split[group_df.loc[idx, group_col]] = "calibration"
                for idx in g_val:
                    group_to_split[group_df.loc[idx, group_col]] = "validation"
                for idx in g_test:
                    group_to_split[group_df.loc[idx, group_col]] = "final_test"
                df["split"] = df[group_col].map(group_to_split).fillna("train")
                return df.reset_index(drop=True)
    train_idx, temp_idx = train_test_split(
        df.index,
        test_size=0.40,
        random_state=seed,
        stratify=y,
    )
    temp_y = y.loc[temp_idx]
    if temp_y.value_counts().min() < 3:
        df.loc[train_idx, "split"] = "train"
        df.loc[temp_idx, "split"] = "validation"
        return df
    calib_idx, hold_idx = train_test_split(
        temp_idx,
        test_size=0.50,
        random_state=seed,
        stratify=temp_y,
    )
    hold_y = y.loc[hold_idx]
    if hold_y.value_counts().min() < 2:
        val_idx, test_idx = hold_idx, []
    else:
        val_idx, test_idx = train_test_split(
            hold_idx,
            test_size=0.50,
            random_state=seed,
            stratify=hold_y,
        )
    df["split"] = "train"
    df.loc[calib_idx, "split"] = "calibration"
    df.loc[val_idx, "split"] = "validation"
    if len(test_idx):
        df.loc[test_idx, "split"] = "final_test"
    return df.reset_index(drop=True)


def _first_value(value: Any) -> Any:
    if isinstance(value, (list, tuple)):
        return value[0]
    try:
        if hasattr(value, "__iter__") and not isinstance(value, (str, bytes)):
            return list(value)[0]
    except Exception:
        pass
    return value


def load_image_pixels(path: str | Path) -> tuple[np.ndarray, dict[str, Any], str, list[str]]:
    path = Path(path)
    log: list[str] = []
    suffix = path.suffix.lower()
    if suffix in {".dcm", ".dicom"}:
        if pydicom is None:
            raise RuntimeError("pydicom is required to read DICOM files.")
        ds = pydicom.dcmread(str(path), force=True)
        arr = ds.pixel_array.astype(np.float32)
        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
        arr = arr * slope + intercept
        photometric = str(getattr(ds, "PhotometricInterpretation", "")).upper()
        if photometric == "MONOCHROME1":
            arr = arr.max() - arr
            log.append("inverted_MONOCHROME1")
        wc = getattr(ds, "WindowCenter", None)
        ww = getattr(ds, "WindowWidth", None)
        if wc is not None and ww is not None:
            wc = float(_first_value(wc))
            ww = float(_first_value(ww))
            lo, hi = wc - ww / 2, wc + ww / 2
            if hi > lo:
                arr = np.clip(arr, lo, hi)
                log.append("applied_dicom_window")
        metadata = {
            "modality": str(getattr(ds, "Modality", "")),
            "view_position": str(getattr(ds, "ViewPosition", "")),
            "photometric_interpretation": photometric,
            "bits_stored": getattr(ds, "BitsStored", None),
            "manufacturer": str(getattr(ds, "Manufacturer", "")),
            "rows": int(getattr(ds, "Rows", arr.shape[0])),
            "columns": int(getattr(ds, "Columns", arr.shape[1])),
            "patient_id_hash": stable_hash(getattr(ds, "PatientID", str(path.parent))),
            "study_uid_hash": stable_hash(getattr(ds, "StudyInstanceUID", str(path))),
        }
        return arr, metadata, "incxr_dicom", log
    img = Image.open(path).convert("L")
    arr = np.asarray(img).astype(np.float32)
    metadata = {
        "rows": int(arr.shape[0]),
        "columns": int(arr.shape[1]),
        "patient_id_hash": stable_hash(path.parent),
        "study_uid_hash": stable_hash(path),
    }
    return arr, metadata, "png_jpeg", log


def robust_normalize(arr: np.ndarray, lo_pct: float = 0.5, hi_pct: float = 99.5) -> np.ndarray:
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    lo, hi = np.percentile(arr, [lo_pct, hi_pct])
    if hi <= lo:
        return np.zeros_like(arr, dtype=np.float32)
    arr = np.clip(arr, lo, hi)
    return ((arr - lo) / (hi - lo)).astype(np.float32)


def quality_checks(arr01: np.ndarray, metadata: dict[str, Any]) -> tuple[float, list[str], bool]:
    flags: list[str] = []
    h, w = arr01.shape
    if h < 256 or w < 256:
        flags.append("low_resolution")
    contrast = float(np.percentile(arr01, 99) - np.percentile(arr01, 1))
    if contrast < 0.08:
        flags.append("low_contrast")
    if float(np.mean(arr01 < 0.01)) > 0.55:
        flags.append("large_black_border_or_empty_area")
    if float(np.mean(arr01 > 0.99)) > 0.25:
        flags.append("large_white_saturation")
    border = np.concatenate([arr01[:10, :].ravel(), arr01[-10:, :].ravel(), arr01[:, :10].ravel(), arr01[:, -10:].ravel()])
    if float(np.mean(border < 0.01)) > 0.80:
        flags.append("black_frame")
    modality = str(metadata.get("modality", "")).upper()
    if modality and modality not in {"DX", "CR", "DR", "OT"}:
        flags.append("wrong_or_unknown_modality")
    view = str(metadata.get("view_position", "")).upper()
    if view and view not in {"PA", "AP"}:
        flags.append("non_frontal_or_unknown_projection")
    score = 1.0
    penalty = {
        "low_resolution": 0.25,
        "low_contrast": 0.30,
        "large_black_border_or_empty_area": 0.25,
        "large_white_saturation": 0.25,
        "black_frame": 0.10,
        "wrong_or_unknown_modality": 0.30,
        "non_frontal_or_unknown_projection": 0.30,
    }
    for flag in flags:
        score -= penalty.get(flag, 0.15)
    score = float(np.clip(score, 0.0, 1.0))
    critical = any(flag in flags for flag in ["wrong_or_unknown_modality", "non_frontal_or_unknown_projection"]) or score < 0.35
    return score, flags, critical


def make_thorax_roi(arr01: np.ndarray) -> tuple[np.ndarray | None, np.ndarray | None, str, list[str]]:
    log: list[str] = []
    h, w = arr01.shape
    body = arr01 > max(0.04, float(np.percentile(arr01, 12)))
    body = ndimage.binary_opening(body, iterations=1)
    body = ndimage.binary_closing(body, iterations=3)
    labeled, n = ndimage.label(body)
    if n == 0:
        return None, None, "invalid", ["roi_no_foreground"]
    objects = ndimage.find_objects(labeled)
    areas = []
    for idx, sl in enumerate(objects, start=1):
        if sl is None:
            areas.append(0)
        else:
            areas.append(int(np.sum(labeled[sl] == idx)))
    component = int(np.argmax(areas) + 1)
    mask = labeled == component
    ys, xs = np.where(mask)
    if len(xs) == 0 or len(ys) == 0:
        return None, None, "invalid", ["roi_empty_component"]
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    pad_y = int(0.04 * h)
    pad_x = int(0.04 * w)
    y0, y1 = max(0, y0 - pad_y), min(h - 1, y1 + pad_y)
    x0, x1 = max(0, x0 - pad_x), min(w - 1, x1 + pad_x)
    if (y1 - y0) < h * 0.35 or (x1 - x0) < w * 0.25:
        return None, mask.astype(np.uint8), "invalid", ["roi_bbox_too_small"]
    crop = arr01[y0 : y1 + 1, x0 : x1 + 1]
    log.append(f"thorax_roi_bbox={x0},{y0},{x1},{y1}")
    return crop, mask.astype(np.uint8), "valid", log


def resize_pad_array(arr01: np.ndarray, target: int) -> np.ndarray:
    arr01 = np.clip(arr01, 0, 1)
    img = Image.fromarray((arr01 * 255).astype(np.uint8), mode="L")
    w, h = img.size
    scale = target / max(w, h)
    new_w, new_h = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    img = img.resize((new_w, new_h), Image.BILINEAR)
    canvas = Image.new("L", (target, target), 0)
    canvas.paste(img, ((target - new_w) // 2, (target - new_h) // 2))
    return np.asarray(canvas).astype(np.float32) / 255.0


def array_to_pil(arr01: np.ndarray) -> Image.Image:
    return Image.fromarray((np.clip(arr01, 0, 1) * 255).astype(np.uint8), mode="L")


def _save_pil_image(img: Image.Image | None, path: Path) -> str | None:
    if img is None:
        return None
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path, compress_level=1)
    return str(path)


def _save_array_as_png(arr: np.ndarray | None, path: Path) -> str | None:
    if arr is None:
        return None
    path.parent.mkdir(parents=True, exist_ok=True)
    if arr.dtype != np.uint8:
        arr = (np.clip(arr, 0, 1) * 255).astype(np.uint8)
    Image.fromarray(arr, mode="L").save(path, compress_level=1)
    return str(path)


def cache_preprocessed_result(result: PreprocessResult, cfg: NotebookConfig) -> PreprocessResult:
    cache_root = Path(cfg.preprocessed_cache_dir or Path(cfg.artifacts_dir) / "preprocessed_cache")
    study_id = stable_hash(f"{result.source_dataset}:{result.study_id}", n=20)
    result.image_full_path = _save_pil_image(result.image_full, cache_root / "full" / f"{study_id}.png")
    result.image_roi_path = _save_pil_image(result.image_roi, cache_root / "roi" / f"{study_id}.png")
    result.image_eva_path = _save_pil_image(result.image_eva, cache_root / "eva" / f"{study_id}.png")
    result.image_raw_preview_path = _save_array_as_png(result.image_raw_preview, cache_root / "preview" / f"{study_id}.png")
    result.lung_mask_path = None
    result.image_full = None
    result.image_roi = None
    result.image_eva = None
    result.image_raw_preview = None
    result.lung_mask = None
    return result


def _load_pil_image(path: str | None) -> Image.Image | None:
    if not path:
        return None
    with Image.open(path) as img:
        return img.convert("L").copy()


def get_result_image_full(result: PreprocessResult) -> Image.Image:
    if result.image_full is not None:
        return result.image_full
    img = _load_pil_image(result.image_full_path)
    if img is None:
        raise RuntimeError(f"Missing cached full image for study {result.study_id}")
    return img


def get_result_image_roi(result: PreprocessResult) -> Image.Image | None:
    if result.image_roi is not None:
        return result.image_roi
    return _load_pil_image(result.image_roi_path)


def get_result_image_eva(result: PreprocessResult) -> Image.Image:
    if result.image_eva is not None:
        return result.image_eva
    img = _load_pil_image(result.image_eva_path)
    if img is None:
        raise RuntimeError(f"Missing cached EVA image for study {result.study_id}")
    return img


def get_result_raw_preview(result: PreprocessResult) -> np.ndarray:
    if result.image_raw_preview is not None:
        return result.image_raw_preview
    img = _load_pil_image(result.image_raw_preview_path)
    if img is None:
        return np.asarray(get_result_image_full(result)).astype(np.float32) / 255.0
    return np.asarray(img).astype(np.float32) / 255.0


def preprocess_one(row: pd.Series | dict[str, Any], cfg: NotebookConfig) -> PreprocessResult:
    path = str(row["path"])
    raw, metadata, source_type, log = load_image_pixels(path)
    arr01 = robust_normalize(raw)
    quality_score, qa_flags, critical = quality_checks(arr01, metadata)
    roi_arr, mask, roi_status, roi_log = make_thorax_roi(arr01)
    qa_flags.extend([flag for flag in roi_log if flag.startswith("roi_")])
    log.extend(roi_log)
    full = resize_pad_array(arr01, cfg.cxr_foundation_full_size)
    eva = resize_pad_array(arr01, cfg.eva_image_size)
    roi_pil = None
    if roi_arr is not None and roi_status == "valid":
        roi_pil = array_to_pil(resize_pad_array(roi_arr, cfg.cxr_foundation_full_size))
    metadata["rows"] = int(raw.shape[0])
    metadata["columns"] = int(raw.shape[1])
    metadata.setdefault("patient_id_hash", row.get("patient_id_hash", stable_hash(path)))
    return PreprocessResult(
        study_id=str(row.get("study_id", stable_hash(path))),
        source_path=path,
        y_attention=int(row["y_attention"]),
        source_dataset=str(row.get("source_dataset", "unknown")),
        image_full=array_to_pil(full),
        image_roi=roi_pil,
        image_eva=array_to_pil(eva),
        image_raw_preview=arr01,
        lung_mask=mask,
        quality_score=quality_score,
        qa_flags=qa_flags,
        metadata=metadata,
        preprocess_log=log,
        source_type=source_type,
        roi_status=roi_status,
        critical_qa=critical,
    )


def preprocess_dataframe(df: pd.DataFrame, cfg: NotebookConfig, limit: int | None = None) -> tuple[list[PreprocessResult], pd.DataFrame]:
    rows = df.head(limit).to_dict("records") if limit is not None else df.to_dict("records")
    results = []
    total = len(rows)
    for idx, row in enumerate(rows, start=1):
        result = preprocess_one(row, cfg)
        if cfg.cache_preprocessed_to_disk:
            result = cache_preprocessed_result(result, cfg)
        results.append(result)
        if cfg.preprocess_progress_every and (idx == 1 or idx % cfg.preprocess_progress_every == 0 or idx == total):
            print(f"Preprocessed {idx}/{total} studies")
    meta = pd.DataFrame([r.to_record() for r in results])
    meta = meta.merge(df[["study_id", "split"]], on="study_id", how="left")
    return results, meta


def coerce_embedding_output(output: Any) -> np.ndarray:
    if isinstance(output, np.ndarray):
        return output.astype(np.float32)
    if hasattr(output, "error") and getattr(output, "error") not in (None, ""):
        raise RuntimeError(f"CXR Foundation returned an embedding error: {getattr(output, 'error')}")
    if hasattr(output, "general_img_emb") and getattr(output, "general_img_emb") is not None:
        return coerce_embedding_output(getattr(output, "general_img_emb"))
    if hasattr(output, "contrastive_img_emb") and getattr(output, "contrastive_img_emb") is not None:
        return coerce_embedding_output(getattr(output, "contrastive_img_emb"))
    if hasattr(output, "contrastive_txt_emb") and getattr(output, "contrastive_txt_emb") is not None:
        return coerce_embedding_output(getattr(output, "contrastive_txt_emb"))
    if isinstance(output, (list, tuple)):
        if len(output) == 0:
            return np.empty((0, 32, 768), dtype=np.float32)
        if any(
            hasattr(x, "general_img_emb")
            or hasattr(x, "contrastive_img_emb")
            or hasattr(x, "contrastive_txt_emb")
            or hasattr(x, "error")
            for x in output
        ):
            return np.asarray([coerce_embedding_output(x) for x in output], dtype=np.float32)
        return np.asarray(output, dtype=np.float32)
    if isinstance(output, dict):
        if output.get("error"):
            raise RuntimeError(f"CXR Foundation returned an embedding error: {output['error']}")
        for key in [
            "general_img_emb",
            "contrastive_img_emb",
            "contrastive_txt_emb",
            "image_embeddings",
            "embeddings",
            "elixr",
            "cxr_model",
        ]:
            if key in output:
                return coerce_embedding_output(output[key])
        arrays = [
            coerce_embedding_output(v)
            for v in output.values()
            if isinstance(v, (list, tuple, np.ndarray, dict))
            or hasattr(v, "general_img_emb")
            or hasattr(v, "contrastive_img_emb")
            or hasattr(v, "contrastive_txt_emb")
        ]
        if arrays:
            return arrays[0]
    raise TypeError(f"Cannot coerce embedding output of type {type(output)!r}")


def pool_tokens(tokens: np.ndarray) -> np.ndarray:
    tokens = np.asarray(tokens, dtype=np.float32)
    if tokens.ndim == 1:
        return tokens
    if tokens.ndim == 2:
        mean = tokens.mean(axis=0)
        maxv = tokens.max(axis=0)
        std = tokens.std(axis=0)
        return np.concatenate([mean, maxv, std]).astype(np.float32)
    if tokens.ndim == 3:
        return np.vstack([pool_tokens(x) for x in tokens])
    raise ValueError(f"Unexpected token shape: {tokens.shape}")


def qa_feature_matrix(meta: pd.DataFrame) -> tuple[np.ndarray, list[str]]:
    flag_counts = meta["qa_flags"].fillna("").apply(lambda s: 0 if not s else len([x for x in str(s).split("|") if x]))
    roi_missing = (meta["roi_status"] != "valid").astype(float)
    critical = meta["critical_qa"].astype(float)
    rows = pd.to_numeric(meta["rows"], errors="coerce").fillna(0)
    cols = pd.to_numeric(meta["columns"], errors="coerce").fillna(0)
    X = np.vstack(
        [
            meta["quality_score"].astype(float).values,
            flag_counts.astype(float).values,
            roi_missing.values,
            critical.values,
            np.log1p(rows.values),
            np.log1p(cols.values),
        ]
    ).T.astype(np.float32)
    names = ["quality_score", "qa_flag_count", "roi_missing", "critical_qa", "log_rows", "log_cols"]
    return X, names


def build_fusion_features(
    full_tokens: np.ndarray,
    roi_tokens: np.ndarray,
    meta: pd.DataFrame,
) -> tuple[np.ndarray, list[str]]:
    full = pool_tokens(full_tokens)
    roi = pool_tokens(roi_tokens)
    qa, qa_names = qa_feature_matrix(meta)
    X = np.concatenate([full, roi, qa], axis=1).astype(np.float32)
    names = (
        [f"full_pool_{i}" for i in range(full.shape[1])]
        + [f"roi_pool_{i}" for i in range(roi.shape[1])]
        + [f"qa_{n}" for n in qa_names]
    )
    return X, names


def make_logistic_pipeline(pca_components: int | None, n_samples: int, n_features: int, seed: int = 42) -> Pipeline:
    steps: list[tuple[str, Any]] = [("scale", StandardScaler())]
    if pca_components is not None:
        n_comp = min(int(pca_components), max(1, n_samples - 1), n_features)
        if n_comp < n_features:
            steps.append(("pca", PCA(n_components=n_comp, random_state=seed)))
    clf = LogisticRegression(
        penalty="elasticnet",
        l1_ratio=0.10,
        solver="saga",
        C=1.0,
        class_weight="balanced",
        max_iter=4000,
        random_state=seed,
        n_jobs=-1,
    )
    steps.append(("clf", clf))
    return Pipeline(steps)


class ConstantProbabilityModel:
    def __init__(self, p: float):
        self.p = float(p)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        p = np.full(len(X), self.p, dtype=np.float32)
        return np.vstack([1 - p, p]).T


class SigmoidCalibratedModel:
    def __init__(self, base_model: Any):
        self.base_model = base_model
        self.calibrator = LogisticRegression(max_iter=1000)
        self.ready = False

    def fit(self, X_calib: np.ndarray, y_calib: np.ndarray) -> "SigmoidCalibratedModel":
        y_calib = np.asarray(y_calib).astype(int)
        p = np.clip(self.base_model.predict_proba(X_calib)[:, 1], 1e-5, 1 - 1e-5)
        if len(y_calib) >= 4 and len(np.unique(y_calib)) == 2:
            logits = np.log(p / (1 - p)).reshape(-1, 1)
            self.calibrator.fit(logits, y_calib)
            self.ready = True
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        p = np.clip(self.base_model.predict_proba(X)[:, 1], 1e-5, 1 - 1e-5)
        if self.ready:
            logits = np.log(p / (1 - p)).reshape(-1, 1)
            p = self.calibrator.predict_proba(logits)[:, 1]
        return np.vstack([1 - p, p]).T


def calibrate_prefit(model: Any, X_calib: np.ndarray, y_calib: np.ndarray, method: str = "sigmoid") -> Any:
    y_calib = np.asarray(y_calib).astype(int)
    if len(y_calib) < 4 or len(np.unique(y_calib)) < 2:
        return model
    return SigmoidCalibratedModel(model).fit(X_calib, y_calib)


def train_logistic_classifier(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_calib: np.ndarray | None = None,
    y_calib: np.ndarray | None = None,
    pca_components: int | None = 256,
    seed: int = 42,
) -> Any:
    y_train = np.asarray(y_train).astype(int)
    if len(np.unique(y_train)) < 2:
        return ConstantProbabilityModel(float(y_train.mean()))
    model = make_logistic_pipeline(pca_components, len(X_train), X_train.shape[1], seed)
    model.fit(X_train, y_train)
    if X_calib is not None and y_calib is not None:
        model = calibrate_prefit(model, X_calib, y_calib, method="sigmoid")
    return model


class TorchMLP(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 256, dropout: float = 0.20):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, max(32, hidden // 4)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(max(32, hidden // 4), 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


def train_torch_mlp(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray | None = None,
    y_val: np.ndarray | None = None,
    hidden: int = 256,
    dropout: float = 0.20,
    epochs: int = 80,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    seed: int = 42,
    device: str = "cpu",
) -> TorchMLP:
    if torch is None:
        raise RuntimeError("torch is required for MLP training.")
    set_seed(seed)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train).astype(np.float32)
    X_val_s = scaler.transform(X_val).astype(np.float32) if X_val is not None else None
    model = TorchMLP(X_train.shape[1], hidden=hidden, dropout=dropout).to(device)
    model.scaler = scaler  # type: ignore[attr-defined]
    x = torch.tensor(X_train_s, dtype=torch.float32, device=device)
    y = torch.tensor(y_train.astype(np.float32), dtype=torch.float32, device=device)
    pos = float(np.sum(y_train == 1))
    neg = float(np.sum(y_train == 0))
    pos_weight = torch.tensor([neg / max(pos, 1.0)], device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_state = None
    best_loss = float("inf")
    patience = 12
    bad = 0
    for _ in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
        loss.backward()
        opt.step()
        val_loss = float(loss.detach().cpu())
        if X_val_s is not None and y_val is not None and len(y_val) > 0:
            model.eval()
            with torch.no_grad():
                xv = torch.tensor(X_val_s, dtype=torch.float32, device=device)
                yv = torch.tensor(y_val.astype(np.float32), dtype=torch.float32, device=device)
                val_loss = float(F.binary_cross_entropy_with_logits(model(xv), yv).detach().cpu())
        if val_loss < best_loss - 1e-5:
            best_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_torch_mlp(model: TorchMLP, X: np.ndarray, device: str = "cpu") -> np.ndarray:
    scaler = getattr(model, "scaler", None)
    Xs = scaler.transform(X).astype(np.float32) if scaler is not None else X.astype(np.float32)
    model.eval()
    with torch.no_grad():
        x = torch.tensor(Xs, dtype=torch.float32, device=device)
        p = torch.sigmoid(model.to(device)(x)).detach().cpu().numpy()
    return p.astype(np.float32)


class PlattCalibrator:
    def __init__(self):
        self.model = LogisticRegression(max_iter=1000)
        self.ready = False

    def fit(self, p: np.ndarray, y: np.ndarray) -> "PlattCalibrator":
        p = np.clip(np.asarray(p), 1e-5, 1 - 1e-5)
        y = np.asarray(y).astype(int)
        if len(y) >= 4 and len(np.unique(y)) == 2:
            logits = np.log(p / (1 - p)).reshape(-1, 1)
            self.model.fit(logits, y)
            self.ready = True
        return self

    def transform(self, p: np.ndarray) -> np.ndarray:
        if not self.ready:
            return np.asarray(p)
        p = np.clip(np.asarray(p), 1e-5, 1 - 1e-5)
        logits = np.log(p / (1 - p)).reshape(-1, 1)
        return self.model.predict_proba(logits)[:, 1]


def predict_proba_any(model: Any, X: np.ndarray, device: str = "cpu") -> np.ndarray:
    if isinstance(model, TorchMLP):
        p = predict_torch_mlp(model, X, device=device)
        return np.vstack([1 - p, p]).T
    return model.predict_proba(X)


def safe_auc(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y).astype(int)
    if len(np.unique(y)) < 2:
        return float("nan")
    return float(roc_auc_score(y, p))


def safe_auprc(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y).astype(int)
    if len(np.unique(y)) < 2:
        return float("nan")
    return float(average_precision_score(y, p))


def expected_calibration_error(y: np.ndarray, p: np.ndarray, bins: int = 10) -> float:
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (p >= lo) & (p < hi if hi < 1 else p <= hi)
        if not np.any(mask):
            continue
        ece += np.mean(mask) * abs(float(np.mean(y[mask])) - float(np.mean(p[mask])))
    return float(ece)


def metrics_summary(y: np.ndarray, p: np.ndarray, threshold: float = 0.5) -> dict[str, float]:
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p) >= threshold).astype(int)
    out = {
        "n": float(len(y)),
        "prevalence": float(np.mean(y)) if len(y) else float("nan"),
        "auroc": safe_auc(y, p),
        "auprc": safe_auprc(y, p),
        "brier": float(brier_score_loss(y, p)) if len(np.unique(y)) > 1 else float("nan"),
        "ece": expected_calibration_error(y, p),
        "balanced_accuracy@0.5": float(balanced_accuracy_score(y, pred)) if len(np.unique(y)) > 1 else float("nan"),
    }
    return out


def wilson_lower_bound(successes: int, total: int, z: float = 1.96) -> float:
    if total <= 0:
        return float("nan")
    phat = successes / total
    denom = 1.0 + z * z / total
    centre = phat + z * z / (2.0 * total)
    margin = z * math.sqrt((phat * (1.0 - phat) + z * z / (4.0 * total)) / total)
    return float(max(0.0, (centre - margin) / denom))


def threshold_report(y: np.ndarray, p: np.ndarray, target_npv: float = 0.99, n_grid: int = 501) -> pd.DataFrame:
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    rows = []
    thresholds = np.unique(np.quantile(p, np.linspace(0, 1, int(n_grid))))
    for t in thresholds:
        selected = p <= t
        n_sel = int(np.sum(selected))
        if n_sel == 0:
            continue
        fn = int(np.sum((y == 1) & selected))
        tn = int(np.sum((y == 0) & selected))
        npv = tn / max(tn + fn, 1)
        npv_ci95_low = wilson_lower_bound(tn, tn + fn, z=1.96)
        coverage = n_sel / len(y)
        rows.append(
            {
                "T_negative": float(t),
                "selected_count": n_sel,
                "TN_count": tn,
                "no_attention_required_coverage": float(coverage),
                "NPV": float(npv),
                "NPV_ci95_low": float(npv_ci95_low),
                "FN_count": fn,
                "FN_per_1000": float(fn / max(n_sel, 1) * 1000.0),
                "N/A_rate_if_positive_threshold_unused": float(1.0 - coverage),
                "meets_target_NPV": bool(npv >= target_npv),
            }
        )
    report = pd.DataFrame(rows)
    if not report.empty:
        report = report.sort_values(["meets_target_NPV", "no_attention_required_coverage"], ascending=[False, False])
    return report.reset_index(drop=True)


def choose_negative_threshold(report: pd.DataFrame, fallback_quantile: float = 0.15) -> float:
    if report.empty:
        return 0.05
    ok = report[report["meets_target_NPV"]]
    if not ok.empty:
        return float(ok.iloc[0]["T_negative"])
    return float(report["T_negative"].quantile(fallback_quantile))


def route_decisions(
    p: np.ndarray,
    meta: pd.DataFrame,
    t_negative: float,
    t_positive: float = 0.80,
    t_quality: float = 0.35,
    ood_score: np.ndarray | None = None,
    t_ood: float = 0.95,
) -> pd.DataFrame:
    rows = []
    for i, prob in enumerate(np.asarray(p)):
        quality = float(meta.iloc[i]["quality_score"])
        critical = bool(meta.iloc[i]["critical_qa"])
        uncertainty = float(1.0 - abs(prob - 0.5) * 2.0)
        ood = float(ood_score[i]) if ood_score is not None else None
        if quality < t_quality or critical:
            route, reason = "N/A", "bad_quality_or_critical_qa"
        elif ood is not None and ood > t_ood:
            route, reason = "N/A", "out_of_distribution"
        elif prob <= t_negative:
            route, reason = "no_attention_required", "confident_no_attention_required"
        elif prob >= t_positive:
            route, reason = "requires_attention", "suspicious_requires_attention"
        elif uncertainty > 0.65:
            route, reason = "N/A", "high_uncertainty"
        else:
            route, reason = "N/A", "gray_zone"
        rows.append(
            {
                "study_id": meta.iloc[i]["study_id"],
                "p_requires_attention": float(prob),
                "quality_score": quality,
                "ood_score": ood,
                "uncertainty_score": uncertainty,
                "route": route,
                "reason": reason,
            }
        )
    return pd.DataFrame(rows)


def transform_bbox_to_padded_square(
    bbox: dict[str, Any] | pd.Series,
    original_h: int,
    original_w: int,
    target: int,
) -> tuple[float, float, float, float]:
    scale = target / max(float(original_w), float(original_h))
    new_w = float(original_w) * scale
    new_h = float(original_h) * scale
    pad_x = (target - new_w) / 2.0
    pad_y = (target - new_h) / 2.0
    x0 = float(bbox["x_min"]) * scale + pad_x
    y0 = float(bbox["y_min"]) * scale + pad_y
    x1 = float(bbox["x_max"]) * scale + pad_x
    y1 = float(bbox["y_max"]) * scale + pad_y
    return (
        float(np.clip(x0, 0, target - 1)),
        float(np.clip(y0, 0, target - 1)),
        float(np.clip(x1, 0, target - 1)),
        float(np.clip(y1, 0, target - 1)),
    )


def bbox_mask_for_result(
    result: PreprocessResult,
    bbox_df: pd.DataFrame,
    target: int,
) -> np.ndarray:
    mask = np.zeros((target, target), dtype=np.float32)
    if bbox_df is None or bbox_df.empty:
        return mask
    raw_preview = get_result_raw_preview(result)
    fallback_rows = int(result.metadata.get("rows", raw_preview.shape[0]))
    fallback_cols = int(result.metadata.get("columns", raw_preview.shape[1]))
    part = bbox_df[bbox_df["study_id"].astype(str) == str(result.study_id)]
    for _, bbox in part.iterrows():
        if not pd.notna(bbox.get("x_min")):
            continue
        class_name = str(bbox.get("class_name", "")).lower()
        if class_name == "no finding":
            continue
        bbox_rows = pd.to_numeric(pd.Series([bbox.get("bbox_original_rows", np.nan)]), errors="coerce").iloc[0]
        bbox_cols = pd.to_numeric(pd.Series([bbox.get("bbox_original_columns", np.nan)]), errors="coerce").iloc[0]
        rows = int(bbox_rows) if pd.notna(bbox_rows) and float(bbox_rows) > 0 else fallback_rows
        cols = int(bbox_cols) if pd.notna(bbox_cols) and float(bbox_cols) > 0 else fallback_cols
        x0, y0, x1, y1 = transform_bbox_to_padded_square(bbox, rows, cols, target)
        x0i, y0i, x1i, y1i = map(lambda v: int(round(v)), [x0, y0, x1, y1])
        if x1i > x0i and y1i > y0i:
            mask[y0i : y1i + 1, x0i : x1i + 1] = 1.0
    return mask


def heatmap_localization_metrics(heatmap: np.ndarray, bbox_mask: np.ndarray) -> dict[str, float]:
    heatmap = np.asarray(heatmap, dtype=np.float32)
    bbox_mask = np.asarray(bbox_mask, dtype=np.float32)
    if heatmap.shape != bbox_mask.shape:
        zoom_y = bbox_mask.shape[0] / heatmap.shape[0]
        zoom_x = bbox_mask.shape[1] / heatmap.shape[1]
        heatmap = ndimage.zoom(heatmap, (zoom_y, zoom_x), order=1)
    heatmap = np.nan_to_num(heatmap, nan=0.0, posinf=0.0, neginf=0.0)
    heatmap = heatmap - heatmap.min()
    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()
    has_bbox = bool(np.sum(bbox_mask) > 0)
    if not has_bbox:
        return {
            "has_bbox": 0.0,
            "pointing_game_hit": float("nan"),
            "energy_inside_bbox": float("nan"),
            "bbox_iou_at_top20pct": float("nan"),
        }
    max_y, max_x = np.unravel_index(int(np.argmax(heatmap)), heatmap.shape)
    pointing = float(bbox_mask[max_y, max_x] > 0)
    energy = float(np.sum(heatmap * bbox_mask) / max(np.sum(heatmap), 1e-8))
    threshold = float(np.quantile(heatmap, 0.80))
    hm_bin = heatmap >= threshold
    bbox_bin = bbox_mask > 0
    inter = float(np.logical_and(hm_bin, bbox_bin).sum())
    union = float(np.logical_or(hm_bin, bbox_bin).sum())
    iou = inter / max(union, 1.0)
    return {
        "has_bbox": 1.0,
        "pointing_game_hit": pointing,
        "energy_inside_bbox": energy,
        "bbox_iou_at_top20pct": iou,
    }


def make_occlusion_heatmap_from_feature_builder(
    result: PreprocessResult,
    base_meta: pd.DataFrame,
    embed_full_fn: Callable[[list[Image.Image]], np.ndarray],
    original_roi_tokens: np.ndarray,
    predict_feature_fn: Callable[[np.ndarray], float],
    grid: int = 8,
    fill_value: int = 0,
) -> np.ndarray:
    full_image = get_result_image_full(result)
    target = full_image.size[0]
    patch = target // grid
    occluded_images = []
    for gy in range(grid):
        for gx in range(grid):
            img = full_image.copy()
            arr = np.asarray(img).copy()
            y0, y1 = gy * patch, target if gy == grid - 1 else (gy + 1) * patch
            x0, x1 = gx * patch, target if gx == grid - 1 else (gx + 1) * patch
            arr[y0:y1, x0:x1] = fill_value
            occluded_images.append(Image.fromarray(arr.astype(np.uint8), mode="L"))
    base_full = embed_full_fn([full_image])
    base_X, _ = build_fusion_features(base_full, original_roi_tokens[None, ...], base_meta)
    base_score = float(predict_feature_fn(base_X)[0])
    occ_full = embed_full_fn(occluded_images)
    roi_repeated = np.repeat(original_roi_tokens[None, ...], len(occluded_images), axis=0)
    meta_rep = pd.concat([base_meta] * len(occluded_images), ignore_index=True)
    occ_X, _ = build_fusion_features(occ_full, roi_repeated, meta_rep)
    occ_scores = np.asarray(predict_feature_fn(occ_X), dtype=np.float32)
    drops = np.clip(base_score - occ_scores, 0, None).reshape(grid, grid)
    if drops.max() > 0:
        drops = drops / drops.max()
    heatmap = ndimage.zoom(drops, (target / grid, target / grid), order=1)
    return heatmap[:target, :target].astype(np.float32)


def fit_ood_model(X_train: np.ndarray) -> dict[str, Any]:
    scaler = StandardScaler().fit(X_train)
    Xs = scaler.transform(X_train)
    nn_model = NearestNeighbors(n_neighbors=min(5, len(Xs))).fit(Xs)
    dists, _ = nn_model.kneighbors(Xs)
    ref = dists[:, -1]
    iso = IsolationForest(random_state=42, contamination="auto").fit(Xs)
    return {"scaler": scaler, "nn": nn_model, "ref95": float(np.percentile(ref, 95)), "iso": iso}


def ood_score(model: dict[str, Any], X: np.ndarray) -> np.ndarray:
    Xs = model["scaler"].transform(X)
    dists, _ = model["nn"].kneighbors(Xs)
    knn = dists[:, -1] / max(model["ref95"], 1e-6)
    iso_raw = -model["iso"].score_samples(Xs)
    iso = (iso_raw - np.percentile(iso_raw, 5)) / max(np.percentile(iso_raw, 95) - np.percentile(iso_raw, 5), 1e-6)
    return np.clip(0.5 * knn + 0.5 * iso, 0, 2)


def image_to_eva_tensor(img: Image.Image, image_size: int = 224) -> torch.Tensor:
    if torch is None:
        raise RuntimeError("torch is required.")
    img = img.convert("RGB").resize((image_size, image_size), Image.BILINEAR)
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = (arr - 0.5) / 0.5
    return torch.tensor(arr.transpose(2, 0, 1), dtype=torch.float32)


def load_real_eva_x_s(project_dir: str, device: str = "cpu", weights_path: str | None = None) -> Any:
    repo_dir = Path(project_dir) / "external" / "EVA-X"
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/hustvl/EVA-X.git", str(repo_dir)], check=True)
    if str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))
    try:
        import timm  # noqa: F401
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==0.9.0"], check=True)
    if weights_path is None:
        try:
            from huggingface_hub import hf_hub_download
        except Exception:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
            from huggingface_hub import hf_hub_download
        weights_path = hf_hub_download(
            repo_id="MapleF/eva_x",
            filename="eva_x_small_patch16_merged520k_mim.pt",
            local_dir=str(Path(project_dir) / "models" / "eva_x"),
        )
    from eva_x import eva_x_small_patch16

    original_torch_load = torch.load

    def torch_load_eva_x_compat(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        try:
            return original_torch_load(*args, **kwargs)
        except TypeError as exc:
            if "weights_only" in str(exc):
                kwargs.pop("weights_only", None)
                return original_torch_load(*args, **kwargs)
            raise

    torch.load = torch_load_eva_x_compat
    try:
        model = eva_x_small_patch16(pretrained=weights_path)
    finally:
        torch.load = original_torch_load
    model.eval().to(device)
    return model


def extract_eva_features_real(
    model: Any,
    results: list[PreprocessResult],
    image_size: int = 224,
    batch_size: int = 8,
    device: str = "cpu",
) -> np.ndarray:
    if torch is None:
        raise RuntimeError("torch is required.")
    feats = []
    model.eval().to(device)
    with torch.no_grad():
        for start in range(0, len(results), batch_size):
            batch_results = results[start : start + batch_size]
            batch = torch.stack([
                image_to_eva_tensor(get_result_image_eva(r), image_size=image_size)
                for r in batch_results
            ]).to(device)
            if hasattr(model, "forward_features"):
                z = model.forward_features(batch)
                if z.ndim == 3:
                    z = z[:, 1:, :].mean(dim=1) if z.shape[1] > 1 else z.mean(dim=1)
            else:
                z = model(batch)
            feats.append(z.detach().float().cpu().numpy())
    return np.concatenate(feats, axis=0).astype(np.float32)


class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int = 4, alpha: float = 8.0, dropout: float = 0.0):
        super().__init__()
        self.base = base
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / max(r, 1)
        self.dropout = nn.Dropout(dropout)
        device = base.weight.device
        dtype = base.weight.dtype
        self.lora_a = nn.Parameter(torch.zeros((r, base.in_features), device=device, dtype=dtype))
        self.lora_b = nn.Parameter(torch.zeros((base.out_features, r), device=device, dtype=dtype))
        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))
        nn.init.zeros_(self.lora_b)
        for p in self.base.parameters():
            p.requires_grad = False

    @property
    def weight(self) -> torch.Tensor:
        return self.base.weight

    @property
    def bias(self) -> torch.Tensor | None:
        return self.base.bias

    @property
    def in_features(self) -> int:
        return int(self.base.in_features)

    @property
    def out_features(self) -> int:
        return int(self.base.out_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = self.base(x)
        update = F.linear(F.linear(self.dropout(x), self.lora_a), self.lora_b) * self.scaling
        return base + update


def inject_lora_last_blocks(
    model: nn.Module,
    n_last_blocks: int = 2,
    r: int = 4,
    alpha: float = 8.0,
    target_names: tuple[str, ...] = ("qkv", "q_proj", "v_proj", "proj", "fc1", "fc2"),
) -> int:
    if torch is None:
        raise RuntimeError("torch is required.")
    for p in model.parameters():
        p.requires_grad = False
    blocks = getattr(model, "blocks", None)
    if blocks is None:
        blocks = [model]
    target_blocks = list(blocks)[-n_last_blocks:]
    replaced = 0

    def replace_in_module(module: nn.Module, prefix: str = "") -> None:
        nonlocal replaced
        for name, child in list(module.named_children()):
            full = f"{prefix}.{name}" if prefix else name
            if isinstance(child, nn.Linear) and any(key in full for key in target_names):
                setattr(module, name, LoRALinear(child, r=r, alpha=alpha))
                replaced += 1
            else:
                replace_in_module(child, full)

    for block in target_blocks:
        replace_in_module(block)
    return replaced


def count_parameters(model: nn.Module) -> dict[str, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": int(total), "trainable": int(trainable)}


def benchmark_predict(fn: Callable[[], Any], repeats: int = 20) -> dict[str, float]:
    times = []
    for _ in range(max(1, repeats)):
        start = time.perf_counter()
        fn()
        times.append(time.perf_counter() - start)
    return {
        "latency_ms_mean": float(np.mean(times) * 1000),
        "latency_ms_p95": float(np.percentile(times, 95) * 1000),
    }


def quantize_torch_mlp(model: TorchMLP) -> Any:
    if torch is None:
        raise RuntimeError("torch is required.")
    model_cpu = model.to("cpu").eval()
    return torch.quantization.quantize_dynamic(model_cpu, {nn.Linear}, dtype=torch.qint8)


def export_json(data: dict[str, Any], path: str | Path) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)
    return str(path)


def split_arrays_by_meta(X: np.ndarray, y: np.ndarray, meta: pd.DataFrame) -> dict[str, tuple[np.ndarray, np.ndarray, np.ndarray]]:
    out = {}
    for split in ["train", "calibration", "validation", "final_test"]:
        mask = meta["split"].fillna("train").values == split
        if np.any(mask):
            out[split] = (X[mask], y[mask], np.where(mask)[0])
    if "validation" not in out:
        out["validation"] = out.get("calibration", out["train"])
    if "calibration" not in out:
        out["calibration"] = out["validation"]
    if "final_test" not in out:
        out["final_test"] = out["validation"]
    return out


def make_model_report(name: str, y: np.ndarray, p: np.ndarray, meta: pd.DataFrame, target_npv: float) -> tuple[dict[str, float], pd.DataFrame, pd.DataFrame]:
    metrics = metrics_summary(y, p)
    thr = threshold_report(y, p, target_npv=target_npv)
    t_neg = choose_negative_threshold(thr)
    routes = route_decisions(p, meta, t_negative=t_neg)
    metrics["selected_T_negative"] = float(t_neg)
    metrics["auto_negative_coverage"] = float(np.mean(routes["route"] == "no_attention_required"))
    metrics["N/A_rate"] = float(np.mean(routes["route"] == "N/A"))
    metrics["requires_attention_rate"] = float(np.mean(routes["route"] == "requires_attention"))
    metrics["model"] = name
    return metrics, thr, routes


def route_metrics(y: np.ndarray, routes: pd.DataFrame) -> dict[str, float]:
    y = np.asarray(y).astype(int)
    auto = routes["route"].values == "no_attention_required"
    manual = routes["route"].values == "N/A"
    attention = routes["route"].values == "requires_attention"
    fn_auto = int(np.sum((y == 1) & auto))
    tn_auto = int(np.sum((y == 0) & auto))
    fp_attention = int(np.sum((y == 0) & attention))
    fn_manual = int(np.sum((y == 1) & manual))
    return {
        "route_n": float(len(y)),
        "auto_negative_coverage": float(np.mean(auto)) if len(y) else float("nan"),
        "N/A_rate": float(np.mean(manual)) if len(y) else float("nan"),
        "requires_attention_rate": float(np.mean(attention)) if len(y) else float("nan"),
        "auto_negative_NPV": float(tn_auto / max(tn_auto + fn_auto, 1)),
        "unsafe_FN_auto_negative": float(fn_auto),
        "unsafe_FN_per_1000_auto_negative": float(fn_auto / max(np.sum(auto), 1) * 1000.0),
        "safe_FN_in_N/A": float(fn_manual),
        "workload_FP_requires_attention": float(fp_attention),
    }


def fixed_threshold_evaluation(
    name: str,
    y: np.ndarray,
    p: np.ndarray,
    meta: pd.DataFrame,
    t_negative: float,
    t_positive: float = 0.80,
    t_ood: float = 0.95,
    ood_score_values: np.ndarray | None = None,
) -> tuple[dict[str, float], pd.DataFrame]:
    routes = route_decisions(
        p,
        meta,
        t_negative=t_negative,
        t_positive=t_positive,
        ood_score=ood_score_values,
        t_ood=t_ood,
    )
    metrics = metrics_summary(y, p)
    metrics.update(route_metrics(y, routes))
    metrics["model"] = name
    metrics["fixed_T_negative"] = float(t_negative)
    return metrics, routes


def bootstrap_ci(
    y: np.ndarray,
    p: np.ndarray,
    metric_fn: Callable[[np.ndarray, np.ndarray], float],
    n_boot: int = 500,
    seed: int = 42,
) -> tuple[float, float, float]:
    y = np.asarray(y)
    p = np.asarray(p)
    if len(y) == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), size=len(y))
        try:
            value = float(metric_fn(y[idx], p[idx]))
        except Exception:
            value = float("nan")
        if np.isfinite(value):
            values.append(value)
    if not values:
        point = float(metric_fn(y, p))
        return point, float("nan"), float("nan")
    point = float(metric_fn(y, p))
    return point, float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))


def npv_at_threshold(y: np.ndarray, p: np.ndarray, t_negative: float) -> float:
    y = np.asarray(y).astype(int)
    p = np.asarray(p)
    selected = p <= t_negative
    if not np.any(selected):
        return float("nan")
    fn = np.sum((y == 1) & selected)
    tn = np.sum((y == 0) & selected)
    return float(tn / max(tn + fn, 1))


def calibration_table(y: np.ndarray, p: np.ndarray, n_bins: int = 10) -> pd.DataFrame:
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    rows = []
    edges = np.linspace(0, 1, n_bins + 1)
    for bin_id, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (p >= lo) & (p < hi if hi < 1 else p <= hi)
        rows.append(
            {
                "bin": bin_id,
                "prob_low": float(lo),
                "prob_high": float(hi),
                "n": int(np.sum(mask)),
                "mean_pred": float(np.mean(p[mask])) if np.any(mask) else float("nan"),
                "empirical_rate": float(np.mean(y[mask])) if np.any(mask) else float("nan"),
            }
        )
    return pd.DataFrame(rows)


def subgroup_route_report(
    y: np.ndarray,
    p: np.ndarray,
    routes: pd.DataFrame,
    meta: pd.DataFrame,
    group_col: str,
) -> pd.DataFrame:
    rows = []
    work = meta.reset_index(drop=True).copy()
    work["_y"] = np.asarray(y).astype(int)
    work["_p"] = np.asarray(p).astype(float)
    work["_route"] = routes["route"].values
    for group_value, part in work.groupby(group_col, dropna=False):
        idx = part.index.values
        sub_routes = routes.iloc[idx].reset_index(drop=True)
        sub_y = part["_y"].values
        sub_p = part["_p"].values
        row = {
            "group_col": group_col,
            "group_value": str(group_value),
            "n": int(len(part)),
            "prevalence": float(np.mean(sub_y)) if len(part) else float("nan"),
            "auroc": safe_auc(sub_y, sub_p),
            "auprc": safe_auprc(sub_y, sub_p),
        }
        row.update(route_metrics(sub_y, sub_routes))
        rows.append(row)
    return pd.DataFrame(rows)


def permutation_group_importance(
    predict_fn: Callable[[np.ndarray], np.ndarray],
    X: np.ndarray,
    y: np.ndarray,
    group_slices: dict[str, slice],
    metric_fn: Callable[[np.ndarray, np.ndarray], float] = safe_auc,
    n_repeats: int = 5,
    seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    base_p = predict_fn(X)
    base_metric = float(metric_fn(y, base_p))
    rows = []
    for group, sl in group_slices.items():
        values = []
        for _ in range(n_repeats):
            Xp = X.copy()
            perm = rng.permutation(len(Xp))
            Xp[:, sl] = Xp[perm, sl]
            values.append(float(metric_fn(y, predict_fn(Xp))))
        rows.append(
            {
                "feature_group": group,
                "base_metric": base_metric,
                "permuted_metric_mean": float(np.nanmean(values)),
                "metric_drop": float(base_metric - np.nanmean(values)),
                "n_repeats": int(n_repeats),
            }
        )
    return pd.DataFrame(rows).sort_values("metric_drop", ascending=False).reset_index(drop=True)

## 4. Runtime Validation and Project Paths

This cell converts the top-level run profile into a typed `NotebookConfig`, mounts Drive when requested, validates that required datasets/tokens are available, and creates artifact/report directories.

- Production defaults use real Kaggle dataset mirrors and real frozen encoders.
- `MAX_STUDIES=5000` controls IN-CXR sample size.
- `MAX_VINDR_STUDIES=5000` controls VinDr/VinBigData sample size.

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and os.environ.get("MOUNT_DRIVE", "0") == "1":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

DEFAULT_PROJECT_DIR = os.environ.get("PROJECT_DIR")
if not DEFAULT_PROJECT_DIR:
    DEFAULT_PROJECT_DIR = (
        "/content/drive/MyDrive/fluoro_mvp_runs/incxr_t4"
        if IN_COLAB and os.environ.get("MOUNT_DRIVE", "0") == "1"
        else ("/content/fluoro_mvp" if IN_COLAB else str(Path.cwd() / "fluoro_mvp_outputs"))
    )

# --- Main switches ---
DATA_ROOT = os.environ.get("IN_CXR_ROOT") or None
LABELS_CSV = os.environ.get("IN_CXR_LABELS_CSV") or None
VINDR_ROOT = os.environ.get("VINDR_ROOT") or None
MAX_STUDIES = int(os.environ.get("MAX_STUDIES", "5000"))
MAX_VINDR_STUDIES = int(os.environ.get("MAX_VINDR_STUDIES", os.environ.get("VINDR_DOWNLOAD_MAX_STUDIES", "5000")))
CXR_FOUNDATION_FULL_SIZE = int(os.environ.get("CXR_FOUNDATION_FULL_SIZE", "512"))
CACHE_PREPROCESSED_TO_DISK = os.environ.get("CACHE_PREPROCESSED_TO_DISK", "1") == "1"
PREPROCESSED_CACHE_DIR = os.environ.get("PREPROCESSED_CACHE_DIR") or None

RUN_REAL_GOOGLE_CXR = os.environ.get("RUN_REAL_GOOGLE_CXR", "1") == "1"
RUN_REAL_EVA_X = os.environ.get("RUN_REAL_EVA_X", "1") == "1"
RUN_EXP2_LORA = os.environ.get("RUN_EXP2_LORA", "0") == "1"
RUN_PRIMARY_TRACK = os.environ.get("RUN_PRIMARY_TRACK", "1") == "1"
RUN_VINDR_TRACK = os.environ.get("RUN_VINDR_TRACK", "0") == "1"
RUN_VINDR_EXP2 = os.environ.get("RUN_VINDR_EXP2", "0") == "1"
RUN_EXP3_CHEXFOUND = os.environ.get("RUN_EXP3_CHEXFOUND", "0") == "1"

cfg = NotebookConfig(
    project_dir=DEFAULT_PROJECT_DIR,
    data_root=DATA_ROOT,
    labels_csv=LABELS_CSV,
    vindr_root=VINDR_ROOT,
    max_studies=MAX_STUDIES,
    max_vindr_studies=MAX_VINDR_STUDIES,
    random_state=42,
    target_npv=0.99,
    max_fn_per_1000=5.0,
    cxr_foundation_full_size=CXR_FOUNDATION_FULL_SIZE,
    eva_image_size=224,
    batch_size=8,
    run_real_google_cxr=RUN_REAL_GOOGLE_CXR,
    run_real_eva_x=RUN_REAL_EVA_X,
    run_exp2_lora=RUN_EXP2_LORA,
    run_primary_track=RUN_PRIMARY_TRACK,
    run_vindr_track=RUN_VINDR_TRACK,
    run_vindr_exp2=RUN_VINDR_EXP2,
    run_exp3_chexfound=RUN_EXP3_CHEXFOUND,
    pca_components=256,
    cache_preprocessed_to_disk=CACHE_PREPROCESSED_TO_DISK,
    preprocessed_cache_dir=PREPROCESSED_CACHE_DIR,
)
missing_inputs = []
if cfg.run_primary_track and not cfg.data_root and not cfg.labels_csv:
    missing_inputs.append("IN_CXR_ROOT or IN_CXR_LABELS_CSV")
if cfg.run_vindr_track and not cfg.vindr_root:
    missing_inputs.append("VINDR_ROOT")
if cfg.run_real_google_cxr and not os.environ.get("HF_TOKEN"):
    missing_inputs.append("HF_TOKEN")
if missing_inputs:
    raise RuntimeError(
        "Production run is missing: "
        + ", ".join(missing_inputs)
        + ". Run the token/download helper cell first, or override the corresponding flags."
    )
if IN_COLAB and cfg.cache_preprocessed_to_disk and str(cfg.preprocessed_cache_dir).startswith("/content/drive"):
    raise RuntimeError(
        "PREPROCESSED_CACHE_DIR points to Google Drive. Use local Colab disk, for example "
        "os.environ['PREPROCESSED_CACHE_DIR']='/content/fluoro_mvp_preprocessed_cache'."
    )
ensure_dirs(cfg)
set_seed(cfg.random_state)
HEAD_EPOCHS = 80
if torch is not None:
    torch.set_num_threads(min(4, os.cpu_count() or 1))

print(asdict(cfg))
print("HEAD_EPOCHS:", HEAD_EPOCHS)

In [ ]:
print("Project directory:", cfg.project_dir)
print("Preprocessed cache directory:", cfg.preprocessed_cache_dir)

## 5. Dataset Loading and Split

For IN-CXR first branch:

- `normal -> y_attention=0 -> no_attention_required`
- `abnormal -> y_attention=1 -> requires_attention`
- `N/A` is not a dataset label; it is produced later by the router.

For the production run, the intended dataset size is **5000 IN-CXR studies**. This keeps the run inside free Colab limits while preserving enough signal for a meaningful first comparison.

The loader accepts:

- folder structure with `normal/` and `abnormal/`;
- CSV with `path,y_attention`.

In [ ]:
if cfg.run_primary_track:
    df = discover_dataset(cfg.data_root, cfg.labels_csv, max_studies=cfg.max_studies, cfg=cfg)
else:
    raise RuntimeError("RUN_PRIMARY_TRACK=0 is not supported in the production notebook. Run the IN-CXR track for model comparison.")
df = make_splits(df, seed=cfg.random_state)

index_path = save_table(df, Path(cfg.artifacts_dir) / "data_index")
print(f"Dataset index saved to: {index_path}")
display(df.head())
display(df["y_attention"].value_counts().rename("count").to_frame())
display(df["split"].value_counts().rename("count").to_frame())

## 6. Detailed Preprocessing

Implemented according to the plan:

- DICOM/PNG/JPEG read path, including the practical IN-CXR PNG mirror.
- DICOM photometric interpretation, rescale/window handling when original DICOM is used.
- Robust clipping and normalization.
- Aspect-ratio-preserving resize/padding.
- Quality checks and QA flags.
- Thorax/lung ROI heuristic for EXP-1 full/ROI fusion.
- Traceability metadata for analysis and backend export.
- A visual audit sheet saved to artifacts for quick review before trusting metrics.
- Disk-backed preprocessed image cache, so full/ROI/EVA inputs do not stay duplicated in RAM.

In [ ]:
results, meta = preprocess_dataframe(df, cfg)
y = meta["y_attention"].values.astype(int)
preproc_path = save_table(meta, Path(cfg.artifacts_dir) / "preprocessing_report")

print(f"Preprocessing report saved to: {preproc_path}")
print("Target vector:", y.shape, "positive_rate=", float(y.mean()))
display(meta.head())
display(meta[["quality_score", "roi_status", "critical_qa"]].describe(include="all"))
display(meta["qa_flags"].value_counts().head(10).rename("count").to_frame())

In [ ]:
# Visual audit: original normalized preview, full model input, and ROI input.
n_show = min(6, len(results))
fig, axes = plt.subplots(n_show, 3, figsize=(9, 3 * n_show))
if n_show == 1:
    axes = np.asarray([axes])
for i, r in enumerate(results[:n_show]):
    axes[i, 0].imshow(get_result_raw_preview(r), cmap="gray")
    axes[i, 0].set_title(f"raw norm | y={r.y_attention}")
    axes[i, 1].imshow(get_result_image_full(r), cmap="gray")
    axes[i, 1].set_title(f"full | q={r.quality_score:.2f}")
    roi_img = get_result_image_roi(r)
    if roi_img is not None:
        axes[i, 2].imshow(roi_img, cmap="gray")
    else:
        axes[i, 2].text(0.5, 0.5, "ROI missing", ha="center", va="center")
    axes[i, 2].set_title(f"ROI: {r.roi_status}")
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()
preview_path = Path(cfg.artifacts_dir) / "previews" / "preprocessing_audit.png"
fig.savefig(preview_path, dpi=160, bbox_inches="tight")
print(f"Saved preview: {preview_path}")
plt.show()

## 7. EXP-1: Google CXR Foundation Full/ROI Embeddings

First branch variants:

- **EXP-1-FIRST-A:** full + ROI embeddings, QA features, elastic-net logistic regression, calibration, N/A-router.
- **EXP-1-FIRST-B:** same features, small bottleneck MLP + dropout, calibration, N/A-router. Use this only if logistic underfits.

Diagnostic path:

- **EXP-1-ABLATION:** full-only, ROI-only, full+ROI, full+ROI+QA. This is not a separate deployment model; it explains whether ROI/QA features really help.

Real model note:

Google CXR Foundation requires accepting Hugging Face terms. The notebook uses the official `clientside.clients.make_hugging_face_client('cxr_model')` path.

### 7.1 EXP-1 Embedding Extraction

This cell loads Google CXR Foundation, computes full-image and ROI token embeddings, and caches them with a dataset fingerprint. Cached embeddings are reused only when the current dataset matches the saved fingerprint.

In [ ]:
def maybe_install_google_cxr_foundation():
    if not cfg.run_real_google_cxr:
        return
    ensure_distribution_min_version("ml-dtypes", "ml-dtypes>=0.5.0", "0.5.0")
    os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
    if not has_module("clientside"):
        pip_install("git+https://github.com/Google-Health/cxr-foundation.git#subdirectory=python")
        ensure_distribution_min_version("ml-dtypes", "ml-dtypes>=0.5.0", "0.5.0")
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login
        login(token=os.environ["HF_TOKEN"])

def normalize_embedding_batch(arr, expected_n):
    arr = coerce_embedding_output(arr)
    if arr.ndim == 2 and expected_n == 1:
        return arr[None, :, :]
    if arr.ndim == 2 and expected_n > 1:
        return arr[:, None, :]
    if arr.ndim == 3:
        return arr
    if arr.ndim == 1:
        return arr[None, None, :]
    raise ValueError(f"Unexpected embedding shape {arr.shape}")

_GOOGLE_CXR_CLIENT = None

def get_google_cxr_client():
    global _GOOGLE_CXR_CLIENT
    if _GOOGLE_CXR_CLIENT is None:
        maybe_install_google_cxr_foundation()
        from clientside.clients import make_hugging_face_client
        _GOOGLE_CXR_CLIENT = make_hugging_face_client("cxr_model")
    return _GOOGLE_CXR_CLIENT

def embed_images_exp1(images, cfg, seed=13):
    if not cfg.run_real_google_cxr:
        raise RuntimeError("RUN_REAL_GOOGLE_CXR=0 disables EXP-1 embeddings. Enable it for the production notebook.")
    cxr_client = get_google_cxr_client()
    out = cxr_client.get_image_embeddings_from_images(images)
    return normalize_embedding_batch(out, len(images)).astype(np.float32)

def extract_exp1_embeddings(results, cfg, prefix="exp1_google"):
    full_path = Path(cfg.artifacts_dir) / "embeddings" / f"{prefix}_full_tokens.npy"
    roi_path = Path(cfg.artifacts_dir) / "embeddings" / f"{prefix}_roi_tokens.npy"
    fingerprint_path = Path(cfg.artifacts_dir) / "embeddings" / f"{prefix}_fingerprint.json"
    fingerprint = stable_hash(
        "|".join(f"{r.study_id}:{r.source_path}:{r.y_attention}" for r in results),
        n=16,
    )
    if full_path.exists() and roi_path.exists() and fingerprint_path.exists():
        cached_full = np.load(full_path)
        cached_roi = np.load(roi_path)
        cached_info = json.loads(fingerprint_path.read_text(encoding="utf-8"))
        cache_ok = (
            cached_full.shape[0] == len(results)
            and cached_roi.shape[0] == len(results)
            and cached_info.get("fingerprint") == fingerprint
        )
        if cache_ok:
            return cached_full, cached_roi
        print("Cached EXP-1 embeddings do not match the current dataset; recomputing.")

    full_batches, roi_batches = [], []
    for start in range(0, len(results), cfg.batch_size):
        batch = results[start : start + cfg.batch_size]
        full_imgs = [get_result_image_full(r) for r in batch]
        roi_imgs = [get_result_image_roi(r) or get_result_image_full(r) for r in batch]
        full_batches.append(embed_images_exp1(full_imgs, cfg, seed=13))
        roi_batches.append(embed_images_exp1(roi_imgs, cfg, seed=17))
    full_tokens = np.concatenate(full_batches, axis=0).astype(np.float32)
    roi_tokens = np.concatenate(roi_batches, axis=0).astype(np.float32)

    np.save(full_path, full_tokens)
    np.save(roi_path, roi_tokens)
    fingerprint_path.write_text(
        json.dumps({"fingerprint": fingerprint, "n": len(results)}, indent=2),
        encoding="utf-8",
    )
    print(f"Saved EXP-1 embeddings: {full_path}, {roi_path}")
    return full_tokens, roi_tokens

full_tokens, roi_tokens = extract_exp1_embeddings(results, cfg, prefix="incxr_exp1_google")
print("EXP-1 full tokens:", full_tokens.shape)
print("EXP-1 ROI tokens:", roi_tokens.shape)

# EXP-1 embeddings are now cached, so the TensorFlow client is no longer needed in RAM/VRAM.
_GOOGLE_CXR_CLIENT = None
try:
    import gc
    import tensorflow as tf
    tf.keras.backend.clear_session()
    gc.collect()
except Exception as exc:
    print("Google CXR cleanup skipped:", exc)

### 7.2 EXP-1-FIRST-A: Calibrated Logistic Head

This is the primary lightweight head for high-dimensional Google CXR Foundation embeddings. It fuses full image, ROI, and QA features, then calibrates risk scores and builds the first threshold/router report.

In [ ]:
# Build pooled full+ROI+QA feature matrix.
X_exp1, exp1_feature_names = build_fusion_features(full_tokens, roi_tokens, meta)
y = meta["y_attention"].values.astype(int)
parts_exp1 = split_arrays_by_meta(X_exp1, y, meta)

# EXP-1-FIRST-A: elastic-net logistic regression.
exp1a = train_logistic_classifier(
    parts_exp1["train"][0],
    parts_exp1["train"][1],
    parts_exp1["calibration"][0],
    parts_exp1["calibration"][1],
    pca_components=cfg.pca_components,
    seed=cfg.random_state,
)
p_exp1a_val = predict_proba_any(exp1a, parts_exp1["validation"][0], device=cfg.device)[:, 1]
exp1a_metrics, exp1a_thr, exp1a_routes = make_model_report(
    "EXP-1-FIRST-A logistic",
    parts_exp1["validation"][1],
    p_exp1a_val,
    meta.iloc[parts_exp1["validation"][2]].reset_index(drop=True),
    cfg.target_npv,
)

print("EXP-1-FIRST-A metrics")
display(pd.DataFrame([exp1a_metrics]))
display(exp1a_thr.head(10))
display(exp1a_routes.head())

save_pickle(exp1a, Path(cfg.artifacts_dir) / "models" / "exp1a_logistic_calibrated.pkl")
exp1a_thr.to_csv(Path(cfg.reports_dir) / "exp1a_threshold_report.csv", index=False)
exp1a_routes.to_csv(Path(cfg.reports_dir) / "exp1a_routes_validation.csv", index=False)

### 7.3 EXP-1 Diagnostic Ablation

This diagnostic checks whether full-image embeddings, ROI embeddings, and QA metadata contribute useful signal. It is for interpretation and model selection sanity, not a separate deployment candidate.

In [ ]:
# EXP-1 stream ablation for interpretation: full-only, ROI-only, full+ROI, full+ROI+QA.
full_pool = pool_tokens(full_tokens)
roi_pool = pool_tokens(roi_tokens)
qa_X, qa_names = qa_feature_matrix(meta)

ablation_sets = {
    "full_only": full_pool,
    "roi_only": roi_pool,
    "full_roi": np.concatenate([full_pool, roi_pool], axis=1),
    "full_roi_qa": X_exp1,
}
ablation_rows = []
for name, X_ab in ablation_sets.items():
    parts = split_arrays_by_meta(X_ab.astype(np.float32), y, meta)
    model = train_logistic_classifier(
        parts["train"][0], parts["train"][1],
        parts["calibration"][0], parts["calibration"][1],
        pca_components=min(128, max(1, parts["train"][0].shape[0] - 1)),
        seed=cfg.random_state,
    )
    p_val = predict_proba_any(model, parts["validation"][0])[:, 1]
    row = metrics_summary(parts["validation"][1], p_val)
    row["feature_set"] = name
    ablation_rows.append(row)

exp1_ablation = pd.DataFrame(ablation_rows).sort_values("auroc", ascending=False)
exp1_ablation.to_csv(Path(cfg.reports_dir) / "exp1_stream_ablation.csv", index=False)
display(exp1_ablation)

### 7.4 EXP-1-FIRST-B: Bottleneck MLP Head

This trains a small regularized neural head on the same frozen EXP-1 features. It is useful when the logistic head underfits, while still avoiding encoder fine-tuning.

In [ ]:
# EXP-1-FIRST-B: small bottleneck MLP on the same features.
hidden = 128 if len(X_exp1) < 200 else 256
exp1b = train_torch_mlp(
    parts_exp1["train"][0],
    parts_exp1["train"][1],
    parts_exp1["calibration"][0],
    parts_exp1["calibration"][1],
    hidden=hidden,
    epochs=HEAD_EPOCHS,
    lr=1e-3,
    weight_decay=1e-4,
    seed=cfg.random_state,
    device=cfg.device,
)

raw_calib = predict_torch_mlp(exp1b, parts_exp1["calibration"][0], device=cfg.device)
exp1b_platt = PlattCalibrator().fit(raw_calib, parts_exp1["calibration"][1])
p_exp1b_val_raw = predict_torch_mlp(exp1b, parts_exp1["validation"][0], device=cfg.device)
p_exp1b_val = exp1b_platt.transform(p_exp1b_val_raw)

exp1b_metrics, exp1b_thr, exp1b_routes = make_model_report(
    "EXP-1-FIRST-B MLP",
    parts_exp1["validation"][1],
    p_exp1b_val,
    meta.iloc[parts_exp1["validation"][2]].reset_index(drop=True),
    cfg.target_npv,
)
print("EXP-1-FIRST-B metrics")
display(pd.DataFrame([exp1b_metrics]))
display(exp1b_thr.head(10))

torch.save({"state_dict": exp1b.state_dict(), "scaler": exp1b.scaler}, Path(cfg.checkpoints_dir) / "exp1b_mlp.pt")
save_pickle(exp1b_platt, Path(cfg.artifacts_dir) / "calibration" / "exp1b_platt.pkl")
exp1b_thr.to_csv(Path(cfg.reports_dir) / "exp1b_threshold_report.csv", index=False)

## 8. EXP-2: EVA-X-S Frozen Encoder + Head

First branch variants:

- **EXP-2-FIRST-A1:** frozen EVA-X-S features + calibrated logistic/linear head.
- **EXP-2-FIRST-A2:** frozen EVA-X-S features + tuned MLP head and conservative threshold policy search.
- **EXP-2-FIRST-B:** optional LoRA/adapters on the last blocks, only if real EVA-X-S loads and memory is stable.

Production mode loads the real EVA-X-S encoder and caches frozen features.

### 8.1 EXP-2 Feature Extraction

This cell loads EVA-X-S, freezes the encoder, extracts one feature vector per study, and stores the feature matrix. The encoder is not fine-tuned in the default first run.

In [ ]:
if cfg.run_real_eva_x:
    eva_model = load_real_eva_x_s(cfg.project_dir, device=cfg.device)
    X_eva = extract_eva_features_real(
        eva_model,
        results,
        image_size=cfg.eva_image_size,
        batch_size=cfg.batch_size,
        device=cfg.device,
    )
else:
    raise RuntimeError("RUN_REAL_EVA_X=0 disables EXP-2. Enable it for the production notebook.")

np.save(Path(cfg.artifacts_dir) / "embeddings" / "exp2_eva_features.npy", X_eva)
print("EXP-2 EVA feature matrix:", X_eva.shape)
y = meta["y_attention"].values.astype(int)
parts_eva = split_arrays_by_meta(X_eva, y, meta)

### 8.2 EXP-2-FIRST-A1: Frozen EVA-X-S Linear Head

This is the strongest low-risk EVA-X-S baseline: frozen encoder features plus a calibrated logistic head.

In [ ]:
# EXP-2-FIRST-A1: frozen EVA-X-S features + linear/logistic head.
exp2a1 = train_logistic_classifier(
    parts_eva["train"][0],
    parts_eva["train"][1],
    parts_eva["calibration"][0],
    parts_eva["calibration"][1],
    pca_components=min(128, max(1, parts_eva["train"][0].shape[0] - 1)),
    seed=cfg.random_state,
)
p_exp2a1_val = predict_proba_any(exp2a1, parts_eva["validation"][0])[:, 1]
exp2a1_metrics, exp2a1_thr, exp2a1_routes = make_model_report(
    "EXP-2-FIRST-A1 EVA frozen logistic",
    parts_eva["validation"][1],
    p_exp2a1_val,
    meta.iloc[parts_eva["validation"][2]].reset_index(drop=True),
    cfg.target_npv,
)
display(pd.DataFrame([exp2a1_metrics]))
display(exp2a1_thr.head(10))
save_pickle(exp2a1, Path(cfg.artifacts_dir) / "models" / "exp2a1_eva_logistic.pkl")

### 8.3 EXP-2-FIRST-A2: EVA MLP Head and Threshold Tuning

This is the main EVA-X-S head-selection block. EVA-X-S features are already cached, so the expensive encoder is not rerun. The cell retrains several MLP heads, calibrates each one, evaluates AUROC/AUPRC/calibration, and tests multiple `T_negative` policies for the `no_attention_required` router.

The default selected threshold policy is conservative: it requires zero false negatives on validation and caps validation auto-negative coverage. This trades some automation volume for a safer MVP router. `EXP2_SELECTION_OBJECTIVE=quality_first` keeps the highest-ranking model by AUROC/AUPRC; `coverage_first` instead prefers the largest safe auto-clear volume among candidates that pass the selected safety policy. The full tuning table is saved, and the selected head is registered for final-test, interpretation, quantization, manifest, and archive export.

In [ ]:
# EVA-X-S MLP tuning grid. Keep this modest: feature extraction is expensive, but head tuning is cheap.
# You can add/remove rows after the first full-dataset run.
EXP2_SELECTED_THRESHOLD_POLICY = os.environ.get("EXP2_SELECTED_THRESHOLD_POLICY", "zero_fn_cap_08pct")
EXP2_SELECTION_OBJECTIVE = os.environ.get("EXP2_SELECTION_OBJECTIVE", "quality_first").lower()
EXP2_MLP_SEARCH_SPACE = [
    {"name": "baseline_h128", "hidden": 128, "dropout": 0.20, "epochs": 80, "lr": 1e-3, "weight_decay": 1e-4, "seed": cfg.random_state},
    {"name": "strong_h256", "hidden": 256, "dropout": 0.20, "epochs": 120, "lr": 8e-4, "weight_decay": 5e-5, "seed": cfg.random_state},
    {"name": "regularized_h256", "hidden": 256, "dropout": 0.30, "epochs": 140, "lr": 8e-4, "weight_decay": 1e-4, "seed": cfg.random_state + 1},
    {"name": "wide_h384", "hidden": 384, "dropout": 0.25, "epochs": 140, "lr": 5e-4, "weight_decay": 5e-5, "seed": cfg.random_state + 2},
    {"name": "compact_low_wd", "hidden": 192, "dropout": 0.15, "epochs": 120, "lr": 1e-3, "weight_decay": 1e-5, "seed": cfg.random_state + 3},
    {"name": "conservative_h256", "hidden": 256, "dropout": 0.35, "epochs": 160, "lr": 5e-4, "weight_decay": 2e-4, "seed": cfg.random_state + 4},
    {"name": "wide_h512", "hidden": 512, "dropout": 0.30, "epochs": 160, "lr": 3e-4, "weight_decay": 1e-4, "seed": cfg.random_state + 5},
    {"name": "strong_h256_seed2", "hidden": 256, "dropout": 0.20, "epochs": 140, "lr": 8e-4, "weight_decay": 5e-5, "seed": cfg.random_state + 6},
]

THRESHOLD_POLICIES = [
    {"name": "target_npv_max_coverage", "require_zero_fn": False, "coverage_cap": 1.00, "min_selected": 10, "min_npv_ci95_low": None, "t_ood": 0.95},
    {"name": "zero_fn_max_coverage", "require_zero_fn": True, "coverage_cap": 1.00, "min_selected": 10, "min_npv_ci95_low": None, "t_ood": 0.95},
    {"name": "zero_fn_cap_10pct", "require_zero_fn": True, "coverage_cap": 0.10, "min_selected": 10, "min_npv_ci95_low": None, "t_ood": 0.95},
    {"name": "zero_fn_cap_08pct", "require_zero_fn": True, "coverage_cap": 0.08, "min_selected": 10, "min_npv_ci95_low": None, "t_ood": 0.95},
    {"name": "zero_fn_cap_08pct_ood90", "require_zero_fn": True, "coverage_cap": 0.08, "min_selected": 10, "min_npv_ci95_low": None, "t_ood": 0.90},
    {"name": "zero_fn_cap_05pct", "require_zero_fn": True, "coverage_cap": 0.05, "min_selected": 5, "min_npv_ci95_low": None, "t_ood": 0.95},
    {"name": "ci_guard_cap_10pct", "require_zero_fn": True, "coverage_cap": 0.10, "min_selected": 20, "min_npv_ci95_low": 0.95, "t_ood": 0.95},
]

def select_threshold_by_policy(thr_report: pd.DataFrame, policy: dict) -> pd.Series | None:
    if thr_report.empty:
        return None
    candidates = thr_report.copy()
    candidates = candidates[candidates["NPV"] >= cfg.target_npv]
    candidates = candidates[candidates["selected_count"] >= int(policy.get("min_selected", 1))]
    candidates = candidates[candidates["no_attention_required_coverage"] <= float(policy.get("coverage_cap", 1.0)) + 1e-12]
    if policy.get("require_zero_fn", False):
        candidates = candidates[candidates["FN_count"] == 0]
    min_ci = policy.get("min_npv_ci95_low")
    if min_ci is not None:
        candidates = candidates[candidates["NPV_ci95_low"] >= float(min_ci)]
    if candidates.empty:
        return None
    candidates = candidates.sort_values(
        ["no_attention_required_coverage", "NPV_ci95_low", "T_negative"],
        ascending=[False, False, False],
    )
    return candidates.iloc[0]

def sort_selected_candidate_rows(rows: pd.DataFrame, objective: str | None = None) -> pd.DataFrame:
    objective = (objective or EXP2_SELECTION_OBJECTIVE or "quality_first").lower()
    rows = rows.copy()
    if objective in {"coverage_first", "safe_coverage_first", "auto_clear_first"}:
        return rows.sort_values(
            ["auto_negative_coverage", "NPV_ci95_low", "auroc", "auprc", "ece"],
            ascending=[False, False, False, False, True],
            na_position="last",
        )
    if objective in {"balanced", "balanced_safe"}:
        rows["_balanced_score"] = (
            rows["auto_negative_coverage"].fillna(0.0) * 2.0
            + rows["auroc"].fillna(0.0)
            + rows["auprc"].fillna(0.0)
            + rows["NPV_ci95_low"].fillna(0.0) * 0.25
            - rows["ece"].fillna(0.0) * 0.25
        )
        return rows.sort_values(
            ["_balanced_score", "auto_negative_coverage", "auroc", "auprc"],
            ascending=[False, False, False, False],
            na_position="last",
        )
    return rows.sort_values(
        ["auroc", "auprc", "auto_negative_coverage", "NPV_ci95_low", "ece"],
        ascending=[False, False, False, False, True],
        na_position="last",
    )

tuning_records = []
exp2_mlp_tuned_models = {}
exp2_mlp_tuned_calibrators = {}
exp2_mlp_tuned_predictions = {}
exp2_mlp_tuned_threshold_reports = {}
exp2_mlp_tuned_routes = {}

val_meta_eva = meta.iloc[parts_eva["validation"][2]].reset_index(drop=True)

for hp in EXP2_MLP_SEARCH_SPACE:
    print("Training EVA MLP head:", hp)
    model = train_torch_mlp(
        parts_eva["train"][0],
        parts_eva["train"][1],
        parts_eva["calibration"][0],
        parts_eva["calibration"][1],
        hidden=int(hp["hidden"]),
        dropout=float(hp["dropout"]),
        epochs=int(hp["epochs"]),
        lr=float(hp["lr"]),
        weight_decay=float(hp["weight_decay"]),
        seed=int(hp["seed"]),
        device=cfg.device,
    )
    raw_calib = predict_torch_mlp(model, parts_eva["calibration"][0], device=cfg.device)
    calibrator = PlattCalibrator().fit(raw_calib, parts_eva["calibration"][1])
    p_raw_val = predict_torch_mlp(model, parts_eva["validation"][0], device=cfg.device)
    p_val = calibrator.transform(p_raw_val)

    base_metrics = metrics_summary(parts_eva["validation"][1], p_val)
    thr_report = threshold_report(parts_eva["validation"][1], p_val, target_npv=cfg.target_npv)

    key = str(hp["name"])
    exp2_mlp_tuned_models[key] = model
    exp2_mlp_tuned_calibrators[key] = calibrator
    exp2_mlp_tuned_predictions[key] = p_val
    exp2_mlp_tuned_threshold_reports[key] = thr_report

    for policy in THRESHOLD_POLICIES:
        selected = select_threshold_by_policy(thr_report, policy)
        if selected is None:
            record = {**base_metrics, **hp}
            record.update({
                "head_name": key,
                "threshold_policy": policy["name"],
                "threshold_policy_selected": False,
                "selected_T_negative": np.nan,
                "auto_negative_coverage": 0.0,
                "N/A_rate": np.nan,
                "requires_attention_rate": np.nan,
                "auto_negative_NPV": np.nan,
                "unsafe_FN_auto_negative": np.nan,
                "unsafe_FN_per_1000_auto_negative": np.nan,
                "NPV_ci95_low": np.nan,
                "selected_t_ood": float(policy.get("t_ood", 0.95)),
            })
            tuning_records.append(record)
            continue

        t_neg = float(selected["T_negative"])
        routes = route_decisions(p_val, val_meta_eva, t_negative=t_neg, t_ood=float(policy.get("t_ood", 0.95)))
        r_metrics = route_metrics(parts_eva["validation"][1], routes)
        record = {**base_metrics, **hp, **r_metrics}
        record.update({
            "head_name": key,
            "threshold_policy": policy["name"],
            "threshold_policy_selected": True,
            "selected_T_negative": t_neg,
            "threshold_selected_count": int(selected["selected_count"]),
            "threshold_validation_NPV": float(selected["NPV"]),
            "threshold_validation_FN_count": int(selected["FN_count"]),
            "NPV_ci95_low": float(selected["NPV_ci95_low"]),
            "selected_t_ood": float(policy.get("t_ood", 0.95)),
            "model": "EXP-2-FIRST-A2-TUNED EVA frozen MLP",
        })
        tuning_records.append(record)
        if policy["name"] == EXP2_SELECTED_THRESHOLD_POLICY:
            exp2_mlp_tuned_routes[key] = routes

exp2_mlp_tuning_results = pd.DataFrame(tuning_records)
exp2_mlp_tuning_results = exp2_mlp_tuning_results.sort_values(
    ["threshold_policy_selected", "threshold_policy", "auroc", "auprc", "auto_negative_coverage"],
    ascending=[False, True, False, False, False],
)
exp2_mlp_tuning_results.to_csv(Path(cfg.reports_dir) / "exp2_mlp_tuning_results.csv", index=False)
export_json(
    {
        "search_space": EXP2_MLP_SEARCH_SPACE,
        "threshold_policies": THRESHOLD_POLICIES,
        "selected_threshold_policy": EXP2_SELECTED_THRESHOLD_POLICY,
        "selection_objective": EXP2_SELECTION_OBJECTIVE,
    },
    Path(cfg.artifacts_dir) / "models" / "exp2_mlp_tuning_config.json",
)
display(exp2_mlp_tuning_results.head(30))

selected_policy_rows = exp2_mlp_tuning_results[
    (exp2_mlp_tuning_results["threshold_policy"] == EXP2_SELECTED_THRESHOLD_POLICY)
    & (exp2_mlp_tuning_results["threshold_policy_selected"])
].copy()
if selected_policy_rows.empty:
    print(f"No rows satisfied {EXP2_SELECTED_THRESHOLD_POLICY}; falling back to any selected threshold policy.")
    selected_policy_rows = exp2_mlp_tuning_results[exp2_mlp_tuning_results["threshold_policy_selected"]].copy()
if selected_policy_rows.empty:
    raise RuntimeError("No valid EVA MLP threshold policy was found. Lower min_selected or coverage constraints.")

selected_policy_rows = sort_selected_candidate_rows(selected_policy_rows, EXP2_SELECTION_OBJECTIVE)
best_tuned_row = selected_policy_rows.iloc[0].to_dict()
exp2a2_tuned_name = str(best_tuned_row["head_name"])
exp2a2_tuned = exp2_mlp_tuned_models[exp2a2_tuned_name]
exp2a2_tuned_platt = exp2_mlp_tuned_calibrators[exp2a2_tuned_name]
p_exp2a2_tuned_val = exp2_mlp_tuned_predictions[exp2a2_tuned_name]
exp2a2_tuned_thr = exp2_mlp_tuned_threshold_reports[exp2a2_tuned_name]
exp2a2_tuned_routes = route_decisions(
    p_exp2a2_tuned_val,
    val_meta_eva,
    t_negative=float(best_tuned_row["selected_T_negative"]),
    t_ood=float(best_tuned_row.get("selected_t_ood", 0.95)),
)

exp2a2_tuned_metrics = metrics_summary(parts_eva["validation"][1], p_exp2a2_tuned_val)
exp2a2_tuned_metrics.update({
    "selected_T_negative": float(best_tuned_row["selected_T_negative"]),
    "auto_negative_coverage": float(best_tuned_row["auto_negative_coverage"]),
    "N/A_rate": float(best_tuned_row["N/A_rate"]),
    "requires_attention_rate": float(best_tuned_row["requires_attention_rate"]),
    "auto_negative_NPV": float(best_tuned_row["auto_negative_NPV"]),
    "unsafe_FN_auto_negative": float(best_tuned_row["unsafe_FN_auto_negative"]),
    "unsafe_FN_per_1000_auto_negative": float(best_tuned_row["unsafe_FN_per_1000_auto_negative"]),
    "threshold_policy": str(best_tuned_row["threshold_policy"]),
    "selected_t_ood": float(best_tuned_row.get("selected_t_ood", 0.95)),
    "head_name": exp2a2_tuned_name,
    "model": "EXP-2-FIRST-A2-TUNED EVA frozen MLP",
})

print("Selected tuned EVA MLP:", exp2a2_tuned_name)
print("Selected threshold policy:", exp2a2_tuned_metrics["threshold_policy"])
print("Selection objective:", EXP2_SELECTION_OBJECTIVE)
display(pd.DataFrame([exp2a2_tuned_metrics]))
display(exp2a2_tuned_thr.head(15))

torch.save({"state_dict": exp2a2_tuned.state_dict(), "scaler": exp2a2_tuned.scaler}, Path(cfg.checkpoints_dir) / "exp2a2_tuned_eva_mlp.pt")
save_pickle(exp2a2_tuned_platt, Path(cfg.artifacts_dir) / "calibration" / "exp2a2_tuned_platt.pkl")
exp2a2_tuned_thr.to_csv(Path(cfg.reports_dir) / "exp2a2_tuned_threshold_report.csv", index=False)
exp2a2_tuned_routes.to_csv(Path(cfg.reports_dir) / "exp2a2_tuned_routes_validation.csv", index=False)

### 8.4 EXP-2-FIRST-B: Optional EVA-X-S LoRA

LoRA is off by default. If enabled, it trains adapters only on the train split, saves a checkpoint after each epoch, calibrates on the calibration split, then runs the same threshold-policy search as the MLP branch. That means a 20- or 40-epoch LoRA run is automatically compared against the tuned MLP with the same safety rules.

In [ ]:
# EXP-2-FIRST-B: optional LoRA/adapters controlled extension.
# This cell defines a real end-to-end training path but does not run unless both flags are true.
class EVAEndToEndClassifier(nn.Module):
    def __init__(self, encoder, feature_dim: int):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def encode(self, x):
        if hasattr(self.encoder, "forward_features"):
            z = self.encoder.forward_features(x)
            if z.ndim == 3:
                z = z[:, 1:, :].mean(dim=1) if z.shape[1] > 1 else z.mean(dim=1)
        else:
            z = self.encoder(x)
        return z

    def forward(self, x):
        return self.head(self.encode(x)).squeeze(-1)

def train_exp2_lora_end_to_end(eva_model, results, labels, cfg, train_indices, epochs=2):
    replaced = inject_lora_last_blocks(eva_model, n_last_blocks=2, r=4, alpha=8.0)
    print(f"LoRA Linear modules replaced: {replaced}")
    eva_model.to(cfg.device)
    sample = image_to_eva_tensor(get_result_image_eva(results[0]), cfg.eva_image_size).unsqueeze(0).to(cfg.device)
    with torch.no_grad():
        if hasattr(eva_model, "forward_features"):
            z = eva_model.forward_features(sample)
            if z.ndim == 3:
                z = z[:, 1:, :].mean(dim=1) if z.shape[1] > 1 else z.mean(dim=1)
        else:
            z = eva_model(sample)
    model = EVAEndToEndClassifier(eva_model, int(z.shape[-1])).to(cfg.device)
    print("EVA LoRA params:", count_parameters(model))
    y_tensor = torch.tensor(labels.astype(np.float32), device=cfg.device)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.device == "cuda"))
    lora_batch_size = int(os.environ.get("EXP2_LORA_BATCH_SIZE", "4"))
    checkpoint_dir = Path(cfg.checkpoints_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    model.train()
    for epoch in range(epochs):
        order = np.random.permutation(np.asarray(train_indices, dtype=int))
        losses = []
        for start in range(0, len(order), lora_batch_size):
            idx = order[start:start + lora_batch_size]
            xb = torch.stack([image_to_eva_tensor(get_result_image_eva(results[int(i)]), cfg.eva_image_size) for i in idx]).to(cfg.device)
            yb = y_tensor[idx]
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(cfg.device == "cuda")):
                loss = F.binary_cross_entropy_with_logits(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        epoch_loss = float(np.mean(losses)) if losses else float("nan")
        epoch_path = checkpoint_dir / f"exp2b_eva_lora_epoch_{epoch + 1}.pt"
        torch.save({"epoch": epoch + 1, "state_dict": model.state_dict(), "loss": epoch_loss}, epoch_path)
        print(f"LoRA epoch {epoch+1}: loss={epoch_loss:.4f}; checkpoint={epoch_path}")
    return model

def predict_lora_end_to_end(model, results, indices, cfg):
    lora_batch_size = int(os.environ.get("EXP2_LORA_BATCH_SIZE", "4"))
    preds = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(indices), lora_batch_size):
            batch_idx = indices[start:start + lora_batch_size]
            xb = torch.stack([image_to_eva_tensor(get_result_image_eva(results[int(i)]), cfg.eva_image_size) for i in batch_idx]).to(cfg.device)
            with torch.cuda.amp.autocast(enabled=(cfg.device == "cuda")):
                logits = model(xb)
            preds.append(torch.sigmoid(logits).detach().float().cpu().numpy())
    return np.concatenate(preds, axis=0).astype(np.float32)

exp2_lora_model = None
exp2_lora_metrics = None
if "EXP2_SELECTION_OBJECTIVE" not in globals():
    EXP2_SELECTION_OBJECTIVE = os.environ.get("EXP2_SELECTION_OBJECTIVE", "quality_first").lower()
if "THRESHOLD_POLICIES" not in globals():
    THRESHOLD_POLICIES = [
        {"name": "target_npv_max_coverage", "require_zero_fn": False, "coverage_cap": 1.00, "min_selected": 10, "min_npv_ci95_low": None, "t_ood": 0.95},
        {"name": "zero_fn_max_coverage", "require_zero_fn": True, "coverage_cap": 1.00, "min_selected": 10, "min_npv_ci95_low": None, "t_ood": 0.95},
        {"name": "zero_fn_cap_10pct", "require_zero_fn": True, "coverage_cap": 0.10, "min_selected": 10, "min_npv_ci95_low": None, "t_ood": 0.95},
        {"name": "zero_fn_cap_08pct", "require_zero_fn": True, "coverage_cap": 0.08, "min_selected": 10, "min_npv_ci95_low": None, "t_ood": 0.95},
        {"name": "zero_fn_cap_08pct_ood90", "require_zero_fn": True, "coverage_cap": 0.08, "min_selected": 10, "min_npv_ci95_low": None, "t_ood": 0.90},
        {"name": "zero_fn_cap_05pct", "require_zero_fn": True, "coverage_cap": 0.05, "min_selected": 5, "min_npv_ci95_low": None, "t_ood": 0.95},
        {"name": "ci_guard_cap_10pct", "require_zero_fn": True, "coverage_cap": 0.10, "min_selected": 20, "min_npv_ci95_low": 0.95, "t_ood": 0.95},
    ]
if "EXP2_SELECTED_THRESHOLD_POLICY" not in globals():
    EXP2_SELECTED_THRESHOLD_POLICY = os.environ.get("EXP2_SELECTED_THRESHOLD_POLICY", "zero_fn_cap_08pct")
if "select_threshold_by_policy" not in globals():
    def select_threshold_by_policy(thr_report: pd.DataFrame, policy: dict) -> pd.Series | None:
        if thr_report.empty:
            return None
        candidates = thr_report.copy()
        candidates = candidates[candidates["NPV"] >= cfg.target_npv]
        candidates = candidates[candidates["selected_count"] >= int(policy.get("min_selected", 1))]
        candidates = candidates[candidates["no_attention_required_coverage"] <= float(policy.get("coverage_cap", 1.0)) + 1e-12]
        if policy.get("require_zero_fn", False):
            candidates = candidates[candidates["FN_count"] == 0]
        min_ci = policy.get("min_npv_ci95_low")
        if min_ci is not None:
            candidates = candidates[candidates["NPV_ci95_low"] >= float(min_ci)]
        if candidates.empty:
            return None
        candidates = candidates.sort_values(
            ["no_attention_required_coverage", "NPV_ci95_low", "T_negative"],
            ascending=[False, False, False],
        )
        return candidates.iloc[0]
if "sort_selected_candidate_rows" not in globals():
    def sort_selected_candidate_rows(rows: pd.DataFrame, objective: str | None = None) -> pd.DataFrame:
        objective = (objective or EXP2_SELECTION_OBJECTIVE or "quality_first").lower()
        rows = rows.copy()
        if objective in {"coverage_first", "safe_coverage_first", "auto_clear_first"}:
            return rows.sort_values(
                ["auto_negative_coverage", "NPV_ci95_low", "auroc", "auprc", "ece"],
                ascending=[False, False, False, False, True],
                na_position="last",
            )
        return rows.sort_values(
            ["auroc", "auprc", "auto_negative_coverage", "NPV_ci95_low", "ece"],
            ascending=[False, False, False, False, True],
            na_position="last",
        )
if cfg.run_real_eva_x and cfg.run_exp2_lora:
    split_to_indices = {
        split: meta.index[meta["split"] == split].to_numpy(dtype=int)
        for split in ["train", "calibration", "validation", "final_test"]
    }
    lora_epochs = int(os.environ.get("EXP2_LORA_EPOCHS", "2"))
    exp2_lora_model = train_exp2_lora_end_to_end(
        eva_model,
        results,
        y,
        cfg,
        train_indices=split_to_indices["train"],
        epochs=lora_epochs,
    )
    ensure_dirs(cfg)
    torch.save(exp2_lora_model.state_dict(), Path(cfg.checkpoints_dir) / "exp2b_eva_lora_final.pt")
    raw_lora_calib = predict_lora_end_to_end(exp2_lora_model, results, split_to_indices["calibration"], cfg)
    exp2_lora_platt = PlattCalibrator().fit(raw_lora_calib, y[split_to_indices["calibration"]])
    raw_lora_val = predict_lora_end_to_end(exp2_lora_model, results, split_to_indices["validation"], cfg)
    p_exp2_lora_val = exp2_lora_platt.transform(raw_lora_val)
    y_lora_val = y[split_to_indices["validation"]]
    val_meta_lora = meta.iloc[split_to_indices["validation"]].reset_index(drop=True)
    lora_descriptor = f"lora_last2_r4_e{lora_epochs}"
    lora_base_metrics = metrics_summary(y_lora_val, p_exp2_lora_val)
    exp2_lora_thr = threshold_report(y_lora_val, p_exp2_lora_val, target_npv=cfg.target_npv)

    lora_policy_records = []
    lora_policy_routes = {}
    for policy in THRESHOLD_POLICIES:
        selected = select_threshold_by_policy(exp2_lora_thr, policy)
        if selected is None:
            record = {**lora_base_metrics}
            record.update({
                "head_name": lora_descriptor,
                "threshold_policy": policy["name"],
                "threshold_policy_selected": False,
                "selected_T_negative": np.nan,
                "auto_negative_coverage": 0.0,
                "N/A_rate": np.nan,
                "requires_attention_rate": np.nan,
                "auto_negative_NPV": np.nan,
                "unsafe_FN_auto_negative": np.nan,
                "unsafe_FN_per_1000_auto_negative": np.nan,
                "NPV_ci95_low": np.nan,
                "selected_t_ood": float(policy.get("t_ood", 0.95)),
                "model": "EXP-2-FIRST-B EVA LoRA",
            })
            lora_policy_records.append(record)
            continue

        t_neg = float(selected["T_negative"])
        routes = route_decisions(p_exp2_lora_val, val_meta_lora, t_negative=t_neg, t_ood=float(policy.get("t_ood", 0.95)))
        r_metrics = route_metrics(y_lora_val, routes)
        record = {**lora_base_metrics, **r_metrics}
        record.update({
            "head_name": lora_descriptor,
            "threshold_policy": policy["name"],
            "threshold_policy_selected": True,
            "selected_T_negative": t_neg,
            "threshold_selected_count": int(selected["selected_count"]),
            "threshold_validation_NPV": float(selected["NPV"]),
            "threshold_validation_FN_count": int(selected["FN_count"]),
            "NPV_ci95_low": float(selected["NPV_ci95_low"]),
            "selected_t_ood": float(policy.get("t_ood", 0.95)),
            "model": "EXP-2-FIRST-B EVA LoRA",
        })
        lora_policy_records.append(record)
        if policy["name"] == EXP2_SELECTED_THRESHOLD_POLICY:
            lora_policy_routes[lora_descriptor] = routes

    exp2_lora_threshold_policy_results = pd.DataFrame(lora_policy_records)
    exp2_lora_threshold_policy_results = exp2_lora_threshold_policy_results.sort_values(
        ["threshold_policy_selected", "threshold_policy", "auroc", "auprc", "auto_negative_coverage"],
        ascending=[False, True, False, False, False],
        na_position="last",
    )

    selected_lora_rows = exp2_lora_threshold_policy_results[
        (exp2_lora_threshold_policy_results["threshold_policy"] == EXP2_SELECTED_THRESHOLD_POLICY)
        & (exp2_lora_threshold_policy_results["threshold_policy_selected"])
    ].copy()
    if selected_lora_rows.empty:
        print(f"No LoRA rows satisfied {EXP2_SELECTED_THRESHOLD_POLICY}; falling back to any selected threshold policy.")
        selected_lora_rows = exp2_lora_threshold_policy_results[
            exp2_lora_threshold_policy_results["threshold_policy_selected"]
        ].copy()
    if selected_lora_rows.empty:
        raise RuntimeError("No valid LoRA threshold policy was found. Lower min_selected or coverage constraints.")

    selected_lora_rows = sort_selected_candidate_rows(selected_lora_rows, EXP2_SELECTION_OBJECTIVE)
    best_lora_row = selected_lora_rows.iloc[0].to_dict()
    exp2_lora_routes = route_decisions(
        p_exp2_lora_val,
        val_meta_lora,
        t_negative=float(best_lora_row["selected_T_negative"]),
        t_ood=float(best_lora_row.get("selected_t_ood", 0.95)),
    )
    exp2_lora_metrics = metrics_summary(y_lora_val, p_exp2_lora_val)
    exp2_lora_metrics.update({
        "selected_T_negative": float(best_lora_row["selected_T_negative"]),
        "auto_negative_coverage": float(best_lora_row["auto_negative_coverage"]),
        "N/A_rate": float(best_lora_row["N/A_rate"]),
        "requires_attention_rate": float(best_lora_row["requires_attention_rate"]),
        "auto_negative_NPV": float(best_lora_row["auto_negative_NPV"]),
        "unsafe_FN_auto_negative": float(best_lora_row["unsafe_FN_auto_negative"]),
        "unsafe_FN_per_1000_auto_negative": float(best_lora_row["unsafe_FN_per_1000_auto_negative"]),
        "threshold_policy": str(best_lora_row["threshold_policy"]),
        "selected_t_ood": float(best_lora_row.get("selected_t_ood", 0.95)),
        "head_name": lora_descriptor,
        "model": "EXP-2-FIRST-B EVA LoRA",
    })
    ensure_dirs(cfg)
    save_pickle(exp2_lora_platt, Path(cfg.artifacts_dir) / "calibration" / "exp2b_eva_lora_platt.pkl")
    exp2_lora_thr.to_csv(Path(cfg.reports_dir) / "exp2b_lora_threshold_report.csv", index=False)
    exp2_lora_threshold_policy_results.to_csv(Path(cfg.reports_dir) / "exp2b_lora_threshold_policy_results.csv", index=False)
    exp2_lora_routes.to_csv(Path(cfg.reports_dir) / "exp2b_lora_routes_validation.csv", index=False)
    export_json(exp2_lora_metrics, Path(cfg.artifacts_dir) / "models" / "exp2b_lora_metrics_validation.json")
    print("Selected LoRA threshold policy:", exp2_lora_metrics["threshold_policy"])
    print("LoRA selection objective:", EXP2_SELECTION_OBJECTIVE)
    display(pd.DataFrame([exp2_lora_metrics]))
    display(exp2_lora_threshold_policy_results.head(20))
    display(exp2_lora_thr.head(10))
else:
    print("RUN_EXP2_LORA=0: skipping optional EVA-X-S LoRA branch.")

## 9. Optional VinDr-CXR 5000-Study Localization Track

VinDr-CXR is not replacing IN-CXR in this notebook. It is used as a second dataset track for the two main model paths:

- sample up to **5000 VinDr/VinBigData studies**;
- map image-level labels to `y_attention`;
- keep bounding boxes for interpretation validation;
- run **EXP-1-FIRST-A** style full/ROI embedding + calibrated logistic head;
- run **EXP-2** EVA-X-S frozen features + calibrated head when `RUN_VINDR_EXP2=1`;
- produce bbox-aware heatmap/attribution diagnostics.

Important data note:

- Official VinDr/VinBigData DICOM data has bbox coordinates in the same pixel space as the source image.
- Public resized PNG mirrors can keep bbox coordinates from the original DICOM. In that case, put metadata such as `images.csv` / `train_meta.csv` with original `Rows` and `Columns` in `VINDR_ROOT`; the loader uses it to scale bboxes correctly.
- Without original-size metadata, classification can still run, but bbox-based interpretation should be treated as invalid.

Product explanation:

IN-CXR checks screening-like triage quality. VinDr-CXR checks whether explanations are spatially plausible because it has radiologist bounding boxes. This gives us two different kinds of evidence without mixing their claims.

### 9.1 VinDr Dataset Loading

This cell is skipped in the first-machine IN-CXR profile. When enabled in a separate profile, it loads VinDr/VinBigData images, labels, metadata, and bounding boxes, then applies the same preprocessing and split logic.

In [ ]:
vindr_ready = False
if cfg.run_vindr_track:
    vindr_df, vindr_bboxes = discover_vindr_dataset(
        cfg.vindr_root,
        max_studies=cfg.max_vindr_studies,
        cfg=cfg,
    )
    vindr_df = make_splits(vindr_df, seed=cfg.random_state)
    vindr_results, vindr_meta = preprocess_dataframe(vindr_df, cfg)
    save_table(vindr_df, Path(cfg.artifacts_dir) / "vindr_data_index")
    save_table(vindr_meta, Path(cfg.artifacts_dir) / "vindr_preprocessing_report")
    vindr_bboxes.to_csv(Path(cfg.artifacts_dir) / "vindr_bboxes.csv", index=False)
    print("VinDr dataframe:", vindr_df.shape)
    print("VinDr bboxes:", vindr_bboxes.shape)
    display(vindr_df["y_attention"].value_counts().rename("count").to_frame())
    display(vindr_df["split"].value_counts().rename("count").to_frame())
    display(vindr_bboxes.head())
    vindr_ready = True
else:
    print("RUN_VINDR_TRACK=0, skipping VinDr-CXR track.")

### 9.2 VinDr Model Runs

When VinDr is enabled, this cell trains the same first-branch heads on the VinDr track and writes a separate VinDr comparison table. These outputs are kept separate from IN-CXR reports.

In [ ]:
if vindr_ready:
    vindr_full_tokens, vindr_roi_tokens = extract_exp1_embeddings(
        vindr_results,
        cfg,
        prefix="vindr_exp1_google",
    )
    X_vindr_exp1, vindr_feature_names = build_fusion_features(vindr_full_tokens, vindr_roi_tokens, vindr_meta)
    y_vindr = vindr_meta["y_attention"].values.astype(int)
    parts_vindr = split_arrays_by_meta(X_vindr_exp1, y_vindr, vindr_meta)

    vindr_exp1 = train_logistic_classifier(
        parts_vindr["train"][0],
        parts_vindr["train"][1],
        parts_vindr["calibration"][0],
        parts_vindr["calibration"][1],
        pca_components=cfg.pca_components,
        seed=cfg.random_state,
    )
    p_vindr_val = predict_proba_any(vindr_exp1, parts_vindr["validation"][0])[:, 1]
    vindr_metrics, vindr_thr, vindr_routes = make_model_report(
        "VinDr EXP-1-FIRST-A localization sanity",
        parts_vindr["validation"][1],
        p_vindr_val,
        vindr_meta.iloc[parts_vindr["validation"][2]].reset_index(drop=True),
        cfg.target_npv,
    )
    save_pickle(vindr_exp1, Path(cfg.artifacts_dir) / "models" / "vindr_exp1a_logistic_calibrated.pkl")
    vindr_thr.to_csv(Path(cfg.reports_dir) / "vindr_exp1_threshold_report.csv", index=False)
    vindr_routes.to_csv(Path(cfg.reports_dir) / "vindr_exp1_routes_validation.csv", index=False)
    display(pd.DataFrame([vindr_metrics]))
    display(vindr_thr.head(10))

    vindr_exp2_metrics = None
    if cfg.run_vindr_exp2:
        if cfg.run_real_eva_x and eva_model is not None:
            X_vindr_eva = extract_eva_features_real(
                eva_model,
                vindr_results,
                image_size=cfg.eva_image_size,
                batch_size=cfg.batch_size,
                device=cfg.device,
            )
        else:
            raise RuntimeError("RUN_REAL_EVA_X=0 disables VinDr EXP-2. Enable it or set RUN_VINDR_EXP2=0.")
        np.save(Path(cfg.artifacts_dir) / "embeddings" / "vindr_exp2_eva_features.npy", X_vindr_eva)
        parts_vindr_eva = split_arrays_by_meta(X_vindr_eva, y_vindr, vindr_meta)
        vindr_exp2 = train_logistic_classifier(
            parts_vindr_eva["train"][0],
            parts_vindr_eva["train"][1],
            parts_vindr_eva["calibration"][0],
            parts_vindr_eva["calibration"][1],
            pca_components=cfg.pca_components,
            seed=cfg.random_state,
        )
        p_vindr_exp2_val = predict_proba_any(vindr_exp2, parts_vindr_eva["validation"][0])[:, 1]
        vindr_exp2_metrics, vindr_exp2_thr, vindr_exp2_routes = make_model_report(
            "VinDr EXP-2 EVA-X-S frozen logistic",
            parts_vindr_eva["validation"][1],
            p_vindr_exp2_val,
            vindr_meta.iloc[parts_vindr_eva["validation"][2]].reset_index(drop=True),
            cfg.target_npv,
        )
        save_pickle(vindr_exp2, Path(cfg.artifacts_dir) / "models" / "vindr_exp2_eva_logistic.pkl")
        vindr_exp2_thr.to_csv(Path(cfg.reports_dir) / "vindr_exp2_threshold_report.csv", index=False)
        vindr_exp2_routes.to_csv(Path(cfg.reports_dir) / "vindr_exp2_routes_validation.csv", index=False)
        display(pd.DataFrame([vindr_exp2_metrics]))
        display(vindr_exp2_thr.head(10))

        vindr_model_comparison = pd.DataFrame([vindr_metrics, vindr_exp2_metrics]).sort_values(
            ["auto_negative_coverage", "auroc"],
            ascending=[False, False],
        )
    else:
        print("RUN_VINDR_EXP2=0, skipping VinDr EXP-2.")
        vindr_model_comparison = pd.DataFrame([vindr_metrics])
    vindr_model_comparison.to_csv(Path(cfg.reports_dir) / "vindr_model_comparison.csv", index=False)
    display(vindr_model_comparison)

### 9.3 VinDr BBox-aware Interpretation

This cell builds occlusion heatmaps and compares them to radiologist bounding boxes. It is the localization sanity check; it does not run for IN-CXR because IN-CXR has no lesion masks or boxes.

In [ ]:
if vindr_ready:
    # BBox-aware occlusion heatmaps for a few abnormal validation cases.
    vindr_interp_dir = Path(cfg.artifacts_dir) / "vindr_interpretation"
    vindr_interp_dir.mkdir(parents=True, exist_ok=True)

    val_indices = parts_vindr["validation"][2]
    candidate_indices = [
        int(i) for i in val_indices
        if y_vindr[int(i)] == 1 and not vindr_bboxes[vindr_bboxes["study_id"].astype(str) == str(vindr_results[int(i)].study_id)].empty
    ]
    if not candidate_indices:
        candidate_indices = [int(i) for i in val_indices[: min(4, len(val_indices))]]

    def predict_vindr_exp1(X_batch):
        return predict_proba_any(vindr_exp1, X_batch)[:, 1]

    def embed_full_for_vindr_heatmap(images):
        return embed_images_exp1(images, cfg, seed=13)

    heatmap_metric_rows = []
    for rank, idx in enumerate(candidate_indices[:4]):
        result = vindr_results[idx]
        base_meta_one = vindr_meta.iloc[[idx]].reset_index(drop=True)
        heatmap = make_occlusion_heatmap_from_feature_builder(
            result,
            base_meta_one,
            embed_full_for_vindr_heatmap,
            vindr_roi_tokens[idx],
            predict_vindr_exp1,
            grid=6,
            fill_value=0,
        )
        result_full = get_result_image_full(result)
        result_roi = get_result_image_roi(result)
        target = result_full.size[0]
        bbox_mask = bbox_mask_for_result(result, vindr_bboxes, target=target)
        loc_metrics = heatmap_localization_metrics(heatmap, bbox_mask)
        loc_metrics.update({"study_id": result.study_id, "rank": rank})
        heatmap_metric_rows.append(loc_metrics)

        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        axes = axes.ravel()
        full_img = np.asarray(result_full)
        axes[0].imshow(get_result_raw_preview(result), cmap="gray")
        axes[0].set_title("Original normalized DICOM")
        axes[1].imshow(full_img, cmap="gray")
        axes[1].set_title("Model full input")
        axes[2].imshow(result_roi if result_roi is not None else result_full, cmap="gray")
        axes[2].set_title(f"ROI stream: {result.roi_status}")
        axes[3].imshow(full_img, cmap="gray")
        axes[3].imshow(heatmap, cmap="magma", alpha=0.55)
        axes[3].contour(bbox_mask, levels=[0.5], colors="cyan", linewidths=2)
        axes[3].set_title("Occlusion attribution + radiologist bbox")
        axes[4].imshow(heatmap, cmap="magma")
        axes[4].contour(bbox_mask, levels=[0.5], colors="cyan", linewidths=2)
        axes[4].set_title(
            "Heatmap only\n"
            f"Energy in bbox={loc_metrics['energy_inside_bbox']:.2f}, "
            f"Pointing hit={loc_metrics['pointing_game_hit']:.0f}"
        )
        axes[5].axis("off")
        explanation = (
            "How to read this panel:\n"
            "1. Magma heatmap = areas where occlusion reduced requires_attention score.\n"
            "2. Cyan contour = VinDr radiologist bbox.\n"
            "3. Energy inside bbox measures how much attribution mass falls inside bbox.\n"
            "4. Pointing game checks whether the hottest point is inside bbox.\n"
            "5. This validates spatial plausibility, not diagnosis correctness."
        )
        axes[5].text(0.0, 1.0, explanation, va="top", fontsize=12)
        for ax in axes[:5]:
            ax.axis("off")
        fig.suptitle(f"VinDr Interpretation Case {rank}: {result.study_id}", fontsize=16)
        plt.tight_layout()
        out = vindr_interp_dir / f"vindr_case_{rank}_{result.study_id}.png"
        fig.savefig(out, dpi=180, bbox_inches="tight")
        plt.show()
        print("Saved:", out)

    vindr_heatmap_metrics = pd.DataFrame(heatmap_metric_rows)
    vindr_heatmap_metrics.to_csv(Path(cfg.reports_dir) / "vindr_heatmap_localization_metrics.csv", index=False)
    display(vindr_heatmap_metrics)

    if not vindr_heatmap_metrics.empty:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        axes[0].bar(vindr_heatmap_metrics["study_id"].astype(str), vindr_heatmap_metrics["energy_inside_bbox"])
        axes[0].set_title("Energy Inside Radiologist BBox")
        axes[0].tick_params(axis="x", rotation=45)
        axes[1].bar(vindr_heatmap_metrics["study_id"].astype(str), vindr_heatmap_metrics["pointing_game_hit"])
        axes[1].set_title("Pointing Game Hit")
        axes[1].tick_params(axis="x", rotation=45)
        axes[2].bar(vindr_heatmap_metrics["study_id"].astype(str), vindr_heatmap_metrics["bbox_iou_at_top20pct"])
        axes[2].set_title("BBox IoU at Top 20% Heatmap")
        axes[2].tick_params(axis="x", rotation=45)
        for ax in axes:
            ax.set_ylim(0, 1)
            ax.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        summary_path = Path(cfg.reports_dir) / "vindr_interpretation_summary.png"
        fig.savefig(summary_path, dpi=180, bbox_inches="tight")
        plt.show()
        print("Saved:", summary_path)

## 10. Optional EXP-3: CheXFound Frozen Reference

EXP-3 is intentionally optional. It is useful as a strong CXR reference, but it should not block the MVP because real CheXFound setup may require repository-specific dependencies and checkpoints.

This notebook keeps the path as:

- clone/check CheXFound repository if enabled;
- use a checkpoint path from `CHEXFOUND_CKPT` if available;
- stop with a clear message until the real checkpoint adapter is implemented.

In [ ]:
exp3_metrics = None
if cfg.run_exp3_chexfound:
    try:
        chexfound_repo = Path(cfg.project_dir) / "external" / "CheXFound"
        chexfound_repo.parent.mkdir(parents=True, exist_ok=True)
        if not chexfound_repo.exists():
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/RPIDIAL/CheXFound.git", str(chexfound_repo)], check=True)
        ckpt = os.environ.get("CHEXFOUND_CKPT")
        if ckpt and Path(ckpt).exists():
            print("CheXFound checkpoint detected:", ckpt)
            print("Real CheXFound feature extraction is repository/checkpoint specific; adapter is intentionally not wired in the first production run.")
        else:
            raise RuntimeError("CHEXFOUND_CKPT must point to a real checkpoint when RUN_EXP3_CHEXFOUND=1.")
        raise NotImplementedError("Real CheXFound feature adapter is not wired yet. Keep RUN_EXP3_CHEXFOUND=0 for the first production run.")
    except Exception as exc:
        print("EXP-3 skipped after setup error:", exc)
        exp3_metrics = None
else:
    print("RUN_EXP3_CHEXFOUND=0, optional EXP-3 skipped.")

## 11. Calibration, N/A-router, Model Comparison

The main product metric is not accuracy. We compare models by:

- auto-negative coverage at target NPV;
- false negatives per 1000 inside the auto-negative zone;
- N/A rate;
- calibration quality.

Postprocessing follows the plan:

- Platt calibration for neural heads where needed.
- Negative threshold search for safe `no_attention_required` routing.
- Positive threshold and gray-zone routing to `requires_attention` / `N/A`.
- QA and OOD gates before automatic clearance.
- `final_test` evaluated with the fixed selected validation threshold, so it stays a held-out check rather than another tuning split.

### 11.1 Validation Comparison

This cell combines EXP-1 and EXP-2 validation reports and registers the selected model for final evaluation. By default it ranks candidates by AUROC/AUPRC first, then safe auto-negative coverage. Set `EXP2_SELECTION_OBJECTIVE=coverage_first` in Run Config when the goal is to maximize safe auto-clear among candidates that satisfy the selected threshold policy.

In [ ]:
model_rows = []
model_registry = {}
MODEL_SELECTION_OBJECTIVE = os.environ.get(
    "EXP2_SELECTION_OBJECTIVE",
    globals().get("EXP2_SELECTION_OBJECTIVE", "quality_first"),
).lower()

def register_candidate(name, metrics_name, model_name, X_name, parts_name, kind, calibrator_name=None):
    if metrics_name not in globals() or model_name not in globals() or X_name not in globals() or parts_name not in globals():
        print(f"Skipping {name}: required variables are not available in this run.")
        return
    metrics = globals()[metrics_name]
    if metrics is None:
        print(f"Skipping {name}: metrics are None.")
        return
    row = dict(metrics)
    row["model"] = name
    model_rows.append(row)
    entry = {
        "model": globals()[model_name],
        "X": globals()[X_name],
        "parts": globals()[parts_name],
        "meta": meta,
        "kind": kind,
    }
    if calibrator_name is not None and calibrator_name in globals():
        entry["calibrator"] = globals()[calibrator_name]
    if "selected_T_negative" in row and pd.notna(row["selected_T_negative"]):
        entry["selected_T_negative"] = float(row["selected_T_negative"])
    if "threshold_policy" in row and pd.notna(row["threshold_policy"]):
        entry["threshold_policy"] = str(row["threshold_policy"])
    if "selected_t_ood" in row and pd.notna(row["selected_t_ood"]):
        entry["selected_t_ood"] = float(row["selected_t_ood"])
    if "head_name" in row and pd.notna(row["head_name"]):
        entry["head_name"] = str(row["head_name"])
    model_registry[name] = entry

def sort_model_candidates(df: pd.DataFrame, objective: str | None = None) -> pd.DataFrame:
    objective = (objective or MODEL_SELECTION_OBJECTIVE or "quality_first").lower()
    out = df.copy()
    if objective in {"coverage_first", "safe_coverage_first", "auto_clear_first"}:
        out["_unsafe_sort"] = out.get("unsafe_FN_auto_negative", pd.Series(np.nan, index=out.index)).fillna(1e9)
        out["_npv_sort"] = out.get("auto_negative_NPV", pd.Series(np.nan, index=out.index)).fillna(-1.0)
        out["_coverage_sort"] = out.get("auto_negative_coverage", pd.Series(np.nan, index=out.index)).fillna(-1.0)
        return out.sort_values(
            ["_unsafe_sort", "_npv_sort", "_coverage_sort", "auroc", "auprc"],
            ascending=[True, False, False, False, False],
            na_position="last",
        ).drop(columns=[c for c in ["_unsafe_sort", "_npv_sort", "_coverage_sort"] if c in out.columns])
    return out.sort_values(
        ["auroc", "auprc", "auto_negative_coverage"],
        ascending=[False, False, False],
        na_position="last",
    )

register_candidate(
    "EXP-1-FIRST-A logistic",
    "exp1a_metrics",
    "exp1a",
    "X_exp1",
    "parts_exp1",
    "sklearn",
)
register_candidate(
    "EXP-1-FIRST-B MLP",
    "exp1b_metrics",
    "exp1b",
    "X_exp1",
    "parts_exp1",
    "torch_mlp",
    "exp1b_platt",
)
register_candidate(
    "EXP-2-FIRST-A1 EVA frozen logistic",
    "exp2a1_metrics",
    "exp2a1",
    "X_eva",
    "parts_eva",
    "sklearn",
)
register_candidate(
    "EXP-2-FIRST-A2-TUNED EVA frozen MLP",
    "exp2a2_tuned_metrics",
    "exp2a2_tuned",
    "X_eva",
    "parts_eva",
    "torch_mlp",
    "exp2a2_tuned_platt",
)
register_candidate(
    "EXP-2-FIRST-A2 EVA frozen MLP",
    "exp2a2_metrics",
    "exp2a2",
    "X_eva",
    "parts_eva",
    "torch_mlp",
    "exp2a2_platt",
)
register_candidate(
    "EXP-2-FIRST-A2-STRONG EVA frozen MLP",
    "exp2a2_strong_metrics",
    "exp2a2_strong",
    "X_eva",
    "parts_eva",
    "torch_mlp",
    "exp2a2_strong_platt",
)
register_candidate(
    "EXP-2-FIRST-B EVA LoRA",
    "exp2_lora_metrics",
    "exp2_lora_model",
    "X_eva",
    "parts_eva",
    "lora_e2e",
    "exp2_lora_platt",
)

if not model_rows:
    raise RuntimeError("No completed model metrics found. Run at least one EXP-1 or EXP-2 head before model comparison.")

primary_model_names = [row["model"] for row in model_rows]
extended_rows = list(model_rows)
if globals().get("exp3_metrics") is not None:
    extended_rows.append(exp3_metrics)
if globals().get("vindr_ready", False) and "vindr_model_comparison" in globals():
    for _, row in vindr_model_comparison.iterrows():
        extended_rows.append(row.to_dict())
model_comparison = pd.DataFrame(model_rows)
model_comparison = sort_model_candidates(model_comparison, MODEL_SELECTION_OBJECTIVE)
model_comparison.to_csv(Path(cfg.reports_dir) / "model_comparison.csv", index=False)
display(model_comparison)

extended_experiment_report = sort_model_candidates(pd.DataFrame(extended_rows), MODEL_SELECTION_OBJECTIVE)
extended_experiment_report.to_csv(Path(cfg.reports_dir) / "extended_experiment_report.csv", index=False)
print("Extended report includes optional EXP-3 and VinDr track when available:")
display(extended_experiment_report)

best_name = str(model_comparison.iloc[0]["model"])
print("Model selection objective:", MODEL_SELECTION_OBJECTIVE)
print("Selected best candidate by validation table:", best_name)
best = model_registry[best_name]

### 11.2 Fixed-threshold Final Test

This cell freezes the selected validation threshold, applies OOD and QA routing, and evaluates the chosen candidate on `final_test` without retuning.

In [ ]:
# OOD-aware routes for the selected model.
best_X = best["X"]
best_parts = best["parts"]
ood = fit_ood_model(best_parts["train"][0])

def score_candidate_split(candidate: dict, split_name: str, ood_model=None):
    parts = candidate["parts"]
    Xs, ys, idx = parts[split_name]
    if candidate["kind"] == "torch_mlp":
        raw = predict_torch_mlp(candidate["model"], Xs, device=cfg.device)
        p = candidate["calibrator"].transform(raw)
    elif candidate["kind"] == "lora_e2e":
        raw = predict_lora_end_to_end(candidate["model"], results, idx, cfg)
        p = candidate["calibrator"].transform(raw)
    else:
        p = predict_proba_any(candidate["model"], Xs)[:, 1]
    ood_s = ood_score(ood_model, Xs) if ood_model is not None else None
    return ys, p, ood_s, idx

y_val_best, p_val_best, ood_val_best, best_val_idx = score_candidate_split(best, "validation", ood_model=ood)
best_thr = threshold_report(y_val_best, p_val_best, target_npv=cfg.target_npv)
if "selected_T_negative" in best and pd.notna(best["selected_T_negative"]):
    selected_T_negative = float(best["selected_T_negative"])
    print("Using validation-selected threshold from model registry:", selected_T_negative)
else:
    selected_T_negative = choose_negative_threshold(best_thr)
    print("Using default max-coverage target-NPV threshold:", selected_T_negative)
selected_t_ood = float(best.get("selected_t_ood", 0.95))
print("Threshold policy:", best.get("threshold_policy", "target_npv_max_coverage"))
print("OOD threshold:", selected_t_ood)
best_val_meta = meta.iloc[best_val_idx].reset_index(drop=True)
best_routes = route_decisions(
    p_val_best,
    best_val_meta,
    t_negative=selected_T_negative,
    ood_score=ood_val_best,
    t_ood=selected_t_ood,
)

y_test_best, p_test_best, ood_test_best, best_test_idx = score_candidate_split(best, "final_test", ood_model=ood)
best_test_meta = meta.iloc[best_test_idx].reset_index(drop=True)
final_test_metrics, final_test_routes = fixed_threshold_evaluation(
    best_name + " final_test",
    y_test_best,
    p_test_best,
    best_test_meta,
    t_negative=selected_T_negative,
    ood_score_values=ood_test_best,
    t_ood=selected_t_ood,
)

print("Best threshold report")
display(best_thr.head(10))
print("Best validation routes")
display(best_routes.head())
print("Final test metrics with fixed validation threshold")
display(pd.DataFrame([final_test_metrics]))
display(final_test_routes.head())
best_routes.to_csv(Path(cfg.reports_dir) / "best_routes_validation.csv", index=False)
best_thr.to_csv(Path(cfg.reports_dir) / "best_threshold_report.csv", index=False)
final_test_routes.to_csv(Path(cfg.reports_dir) / "best_routes_final_test.csv", index=False)
pd.DataFrame([final_test_metrics]).to_csv(Path(cfg.reports_dir) / "best_final_test_metrics.csv", index=False)

# Reporting-only final-test check for every registered candidate. This does not retune
# thresholds and does not change the selected model; it helps compare LoRA vs MLP honestly.
all_candidate_final_rows = []
for candidate_name, candidate in model_registry.items():
    if "selected_T_negative" not in candidate:
        print(f"Skipping final-test candidate report for {candidate_name}: no validation-selected threshold.")
        continue
    try:
        candidate_ood = fit_ood_model(candidate["parts"]["train"][0])
        yy, pp, ood_values, idx = score_candidate_split(candidate, "final_test", ood_model=candidate_ood)
        candidate_meta = meta.iloc[idx].reset_index(drop=True)
        candidate_metrics, _ = fixed_threshold_evaluation(
            candidate_name + " final_test",
            yy,
            pp,
            candidate_meta,
            t_negative=float(candidate["selected_T_negative"]),
            ood_score_values=ood_values,
            t_ood=float(candidate.get("selected_t_ood", 0.95)),
        )
        candidate_metrics["candidate_model"] = candidate_name
        candidate_metrics["threshold_policy"] = candidate.get("threshold_policy", "target_npv_max_coverage")
        candidate_metrics["selected_t_ood"] = float(candidate.get("selected_t_ood", 0.95))
        candidate_metrics["selected_T_negative"] = float(candidate["selected_T_negative"])
        candidate_metrics["head_name"] = candidate.get("head_name")
        all_candidate_final_rows.append(candidate_metrics)
    except Exception as exc:
        print(f"Candidate final-test report failed for {candidate_name}: {exc}")

all_candidates_final_test = pd.DataFrame(all_candidate_final_rows)
if not all_candidates_final_test.empty:
    all_candidates_final_test = sort_model_candidates(all_candidates_final_test, MODEL_SELECTION_OBJECTIVE)
    all_candidates_final_test.to_csv(Path(cfg.reports_dir) / "all_candidates_final_test_metrics.csv", index=False)
    print("All-candidate final-test metrics with fixed validation thresholds:")
    display(all_candidates_final_test)

## 12. Interpretation and Case Review

Interpretation here is not a medical diagnosis. It is a safety/debug artifact:

- show original normalized image;
- show model input;
- show ROI;
- show route reason and QA flags;
- show whether full/ROI/QA fusion helped.

Current interpretation level:

- EXP-1 is interpretable through stream ablation and feature-group permutation: full image vs ROI vs QA.
- EXP-2 is less transparent in frozen-feature mode; this notebook reports feature-level permutation and case review. Strong clinical heatmaps/Grad-CAM should be added only after the real EVA-X-S encoder path is stable.

Important limitation:

IN-CXR first branch is a **binary normal/abnormal dataset without lesion masks/bounding boxes**. That does not make heatmaps impossible: Grad-CAM, attention rollout, occlusion maps, or token attribution can still produce model-attribution heatmaps for the binary decision. But these heatmaps are **not validated localization**. They can show where the model is sensitive, not prove where the pathology is.

Therefore first-branch interpretation focuses on auditable evidence we can defend:

- preprocessing/ROI visual audit;
- route reason and QA flags;
- full vs ROI vs QA ablation;
- permutation importance by feature group;
- qualitative attribution heatmaps only as optional diagnostics after the real encoder path is stable.

### 12.1 Case Review Panels

This cell renders a few routed cases with original normalized image, model input, ROI input, probability, route, and QA context. These panels are for audit and product review.

In [ ]:
def render_case(result, route_row=None):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(get_result_raw_preview(result), cmap="gray")
    axes[0].set_title("Original normalized")
    full_img = get_result_image_full(result)
    roi_img = get_result_image_roi(result)
    axes[1].imshow(full_img, cmap="gray")
    axes[1].set_title("Full input")
    if roi_img is not None:
        axes[2].imshow(roi_img, cmap="gray")
    else:
        axes[2].text(0.5, 0.5, "ROI missing", ha="center", va="center")
    title = f"ROI: {result.roi_status}"
    if route_row is not None:
        title += f"\nroute={route_row['route']} | p={route_row['p_requires_attention']:.3f}"
    axes[2].set_title(title)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    return fig

review_dir = Path(cfg.artifacts_dir) / "case_review"
review_dir.mkdir(parents=True, exist_ok=True)

# Render a few routed cases.
for j, row in best_routes.head(min(6, len(best_routes))).iterrows():
    study_id = row["study_id"]
    result_idx = next(i for i, r in enumerate(results) if r.study_id == study_id)
    fig = render_case(results[result_idx], row)
    safe_route = str(row["route"]).replace("/", "_")
    out = review_dir / f"case_{j}_{safe_route}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

if "exp1_ablation" in globals():
    display(exp1_ablation)
else:
    print("EXP-1 ablation is not available in this run; skipping EXP-1 ablation display.")

### 12.2 Feature-group Interpretation

This cell estimates which feature groups matter by permutation: full-image stream, ROI stream, QA metadata, and EVA frozen features.

In [ ]:
# Feature-group interpretation for EXP-1 selected feature set, when EXP-1 was run.
if all(name in globals() for name in ["full_tokens", "roi_tokens", "exp1a", "parts_exp1"]):
    full_dim = pool_tokens(full_tokens).shape[1]
    roi_dim = pool_tokens(roi_tokens).shape[1]
    qa_dim = qa_feature_matrix(meta)[0].shape[1]
    exp1_group_slices = {
        "full_embedding": slice(0, full_dim),
        "roi_embedding": slice(full_dim, full_dim + roi_dim),
        "qa_metadata": slice(full_dim + roi_dim, full_dim + roi_dim + qa_dim),
    }

    def predict_exp1a_group(X_batch):
        return predict_proba_any(exp1a, X_batch)[:, 1]

    exp1_group_importance = permutation_group_importance(
        predict_exp1a_group,
        parts_exp1["validation"][0],
        parts_exp1["validation"][1],
        exp1_group_slices,
        n_repeats=5,
        seed=cfg.random_state,
    )
    exp1_group_importance.to_csv(Path(cfg.reports_dir) / "exp1_group_importance.csv", index=False)
    display(exp1_group_importance)
else:
    print("EXP-1 feature-group interpretation skipped: EXP-1 variables are not available in this run.")

# Coarse EXP-2 interpretation: permutation of all frozen EVA features.
if "parts_eva" in globals():
    def predict_exp2_group(X_batch):
        if best["kind"] == "torch_mlp" and best.get("X") is X_eva:
            raw = predict_torch_mlp(best["model"], X_batch, device=cfg.device)
            return best["calibrator"].transform(raw)
        if best["kind"] == "sklearn" and best.get("X") is X_eva:
            return predict_proba_any(best["model"], X_batch)[:, 1]
        if "exp2a1" in globals():
            return predict_proba_any(exp2a1, X_batch)[:, 1]
        raise RuntimeError("No EXP-2 predictor is available for group interpretation.")

    exp2_group_importance = permutation_group_importance(
        predict_exp2_group,
        parts_eva["validation"][0],
        parts_eva["validation"][1],
        {"eva_frozen_features": slice(0, parts_eva["validation"][0].shape[1])},
        n_repeats=5,
        seed=cfg.random_state,
    )
    exp2_group_importance.to_csv(Path(cfg.reports_dir) / "exp2_group_importance.csv", index=False)
    display(exp2_group_importance)
else:
    print("EXP-2 feature-group interpretation skipped: parts_eva is not available.")

## 13. Statistics and Error Analysis

Route-aware error analysis:

- false negative in `no_attention_required` is the critical unsafe error;
- false negative in `N/A` is less dangerous because it still goes to manual review;
- false positive in `requires_attention` increases doctor workload but is safer than an unsafe auto-negative.

### 13.1 Route-aware Error Analysis

This cell separates unsafe false negatives in `no_attention_required` from safer manual-review false negatives in `N/A`, which matches the product risk model.

In [ ]:
val_truth = pd.DataFrame({
    "study_id": meta.iloc[best_val_idx]["study_id"].values,
    "y_attention": y_val_best,
    "p_requires_attention": p_val_best,
})
error_analysis = best_routes.merge(val_truth, on=["study_id", "p_requires_attention"], how="left")
error_analysis["unsafe_false_negative"] = (
    (error_analysis["route"] == "no_attention_required") & (error_analysis["y_attention"] == 1)
)
error_analysis["safe_manual_review_false_negative"] = (
    (error_analysis["route"] == "N/A") & (error_analysis["y_attention"] == 1)
)
error_analysis["workload_false_positive"] = (
    (error_analysis["route"] == "requires_attention") & (error_analysis["y_attention"] == 0)
)
error_analysis.to_csv(Path(cfg.reports_dir) / "error_analysis_validation.csv", index=False)
display(error_analysis)
display(error_analysis[["unsafe_false_negative", "safe_manual_review_false_negative", "workload_false_positive"]].sum().rename("count").to_frame())

### 13.2 Calibration, Confidence Intervals, and Subgroups

This cell writes reliability curves, bootstrap confidence intervals, and subgroup route reports using available QA/source metadata.

In [ ]:
# Calibration tables, bootstrap confidence intervals, and subgroup analysis.
val_calibration = calibration_table(y_val_best, p_val_best, n_bins=10)
test_calibration = calibration_table(y_test_best, p_test_best, n_bins=10)
val_calibration.to_csv(Path(cfg.reports_dir) / "calibration_table_validation.csv", index=False)
test_calibration.to_csv(Path(cfg.reports_dir) / "calibration_table_final_test.csv", index=False)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
for table, label in [(val_calibration, "validation"), (test_calibration, "final_test")]:
    nonempty = table.dropna(subset=["mean_pred", "empirical_rate"])
    ax.plot(nonempty["mean_pred"], nonempty["empirical_rate"], marker="o", label=label)
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_xlabel("Mean predicted risk")
ax.set_ylabel("Empirical requires_attention rate")
ax.set_title("Reliability diagram")
ax.legend()
calibration_plot_path = Path(cfg.reports_dir) / "reliability_diagram.png"
fig.savefig(calibration_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", calibration_plot_path)

ci_rows = []
ci_specs = {
    "AUROC": safe_auc,
    "AUPRC": safe_auprc,
    "Brier": lambda yy, pp: brier_score_loss(yy, pp),
    "NPV@selected_T_negative": lambda yy, pp: npv_at_threshold(yy, pp, selected_T_negative),
}
for split_name, yy, pp in [
    ("validation", y_val_best, p_val_best),
    ("final_test", y_test_best, p_test_best),
]:
    for metric_name, fn in ci_specs.items():
        point, lo, hi = bootstrap_ci(yy, pp, fn, n_boot=1000, seed=cfg.random_state)
        ci_rows.append({"split": split_name, "metric": metric_name, "point": point, "ci95_low": lo, "ci95_high": hi})
ci_report = pd.DataFrame(ci_rows)
ci_report.to_csv(Path(cfg.reports_dir) / "bootstrap_ci_report.csv", index=False)
display(ci_report)

# Simple subgroup checks available before real clinical metadata arrives.
best_val_meta_for_groups = best_val_meta.copy()
if best_val_meta_for_groups["quality_score"].nunique() > 1:
    best_val_meta_for_groups["quality_bin"] = pd.qcut(
        best_val_meta_for_groups["quality_score"].rank(method="first"),
        q=min(3, len(best_val_meta_for_groups)),
        duplicates="drop",
    ).astype(str)
else:
    best_val_meta_for_groups["quality_bin"] = "all_same_quality"

subgroup_reports = []
for col in ["roi_status", "quality_bin", "source_type"]:
    if col in best_val_meta_for_groups.columns:
        subgroup_reports.append(subgroup_route_report(y_val_best, p_val_best, best_routes, best_val_meta_for_groups, col))
subgroup_report = pd.concat(subgroup_reports, ignore_index=True) if subgroup_reports else pd.DataFrame()
subgroup_report.to_csv(Path(cfg.reports_dir) / "subgroup_route_report_validation.csv", index=False)
display(subgroup_report)

## 14. Quantization Candidate for Best Head

Quantization is a deployment check, not a way to improve research metrics. After quantization we must re-check calibration/thresholds.

In [ ]:
quant_report = {
    "best_model": best_name,
    "threshold_policy": best.get("threshold_policy", "target_npv_max_coverage"),
    "selected_t_ood": best.get("selected_t_ood", 0.95),
}
if best["kind"] == "torch_mlp":
    X_sample = best_parts["validation"][0][: min(16, len(best_parts["validation"][0]))]
    fp_bench = benchmark_predict(lambda: predict_torch_mlp(best["model"], X_sample, device="cpu"), repeats=20)
    q_model = quantize_torch_mlp(best["model"])
    scaler = getattr(best["model"], "scaler", None)
    def q_predict():
        Xs = scaler.transform(X_sample).astype(np.float32) if scaler is not None else X_sample.astype(np.float32)
        with torch.no_grad():
            return torch.sigmoid(q_model(torch.tensor(Xs))).numpy()
    q_bench = benchmark_predict(q_predict, repeats=20)
    quant_report.update({"fp32_head": fp_bench, "int8_dynamic_head": q_bench})
    torch.save(q_model.state_dict(), Path(cfg.artifacts_dir) / "quantization" / "best_head_dynamic_int8.pt")
elif best["kind"] == "lora_e2e":
    sample_idx = best_parts["validation"][2][: min(8, len(best_parts["validation"][2]))]
    quant_report["note"] = "Best model is EVA-X-S LoRA end-to-end; dynamic head-only quantization is not applied in this notebook."
    quant_report["lora_predict"] = benchmark_predict(
        lambda: predict_lora_end_to_end(best["model"], results, sample_idx, cfg),
        repeats=5,
    )
else:
    X_sample = best_parts["validation"][0][: min(16, len(best_parts["validation"][0]))]
    quant_report["note"] = "Best model is sklearn/logistic; quantization is not needed for this shallow head."
    quant_report["sklearn_predict"] = benchmark_predict(lambda: best["model"].predict_proba(X_sample), repeats=20)

quant_path = export_json(quant_report, Path(cfg.artifacts_dir) / "quantization" / "quantization_report.json")
print("Quantization report:", quant_path)
print(json.dumps(quant_report, indent=2))

## 15. Export Backend Artifacts

This cell saves a compact manifest with model choice, preprocessing version, reports, and router artifacts. Backend should reproduce exactly this logic after the research notebook is accepted.

In [ ]:
manifest = {
    "project": "fluoro_cxr_safety_first_mvp",
    "created_at": pd.Timestamp.utcnow().isoformat(),
    "task": "no_attention_required / requires_attention / N/A",
    "dataset_mode": "real",
    "datasets": {
        "primary_quality": "IN-CXR/local screening-like normal-abnormal" if cfg.run_primary_track else "disabled",
        "localization_sanity": "VinDr/VinBigData 5000-study bbox-aware track" if cfg.run_vindr_track else "disabled",
    },
    "run_primary_track": bool(cfg.run_primary_track),
    "run_vindr_exp2": bool(cfg.run_vindr_exp2),
    "best_model": best_name,
    "best_threshold_policy": best.get("threshold_policy", "target_npv_max_coverage"),
    "best_t_ood": best.get("selected_t_ood", 0.95),
    "best_head_name": best.get("head_name"),
    "optional_exp3_enabled": bool(cfg.run_exp3_chexfound),
    "target_npv": cfg.target_npv,
    "preprocessing_version": "preproc_v1",
    "artifacts_dir": cfg.artifacts_dir,
    "reports_dir": cfg.reports_dir,
    "model_comparison": model_comparison.to_dict(orient="records"),
    "extended_experiment_report": extended_experiment_report.to_dict(orient="records"),
}
manifest_path = export_json(manifest, Path(cfg.artifacts_dir) / "manifest.json")
router_path = export_json(
    {
        "best_model": best_name,
        "threshold_policy": best.get("threshold_policy", "target_npv_max_coverage"),
        "threshold_table": best_thr.head(20).to_dict(orient="records"),
        "selected_T_negative": float(selected_T_negative),
        "selected_t_ood": float(selected_t_ood),
        "router_logic": "quality/OOD checks -> T_negative auto-negative -> T_positive requires_attention -> gray-zone N/A",
    },
    Path(cfg.artifacts_dir) / "router" / "router_config.json",
)
print("Manifest:", manifest_path)
print("Router config:", router_path)

## 16. Final Sanity Checks

These checks are intentionally simple but cover the critical chain:

data -> image preprocessing -> ROI/QA -> enabled experiments -> calibration -> router -> artifacts.

In [ ]:
assert len(df) > 0, "Dataset index is empty"
assert len(results) == len(meta), "Preprocessing results/meta mismatch"
assert set(meta["roi_status"]).issubset({"valid", "invalid", "not_available"})
if "X_exp1" in globals():
    assert X_exp1.shape[0] == len(meta)
if "X_eva" in globals():
    assert X_eva.shape[0] == len(meta)
assert "X_exp1" in globals() or "X_eva" in globals(), "No primary feature matrix is available"
assert not model_comparison.empty
assert not extended_experiment_report.empty
assert best_routes["route"].isin(["no_attention_required", "requires_attention", "N/A"]).all()
assert Path(cfg.artifacts_dir, "manifest.json").exists()
assert Path(cfg.reports_dir, "model_comparison.csv").exists()
if cfg.run_vindr_track and vindr_ready:
    assert Path(cfg.reports_dir, "vindr_heatmap_localization_metrics.csv").exists()
    assert Path(cfg.reports_dir, "vindr_interpretation_summary.png").exists()
    assert Path(cfg.reports_dir, "vindr_model_comparison.csv").exists()

print("FINAL SANITY CHECKS PASSED")
print("Artifacts:", cfg.artifacts_dir)
print("Reports:", cfg.reports_dir)